In [ ]:
# Notebook 1/1 — Full Analysis: Modelling, Evaluation, Explainability, Calibration, Temporal Robustness & Final Artefacts
#
# Purpose:
# Run the complete final analysis pipeline for the dissertation using the validated paired
# retracted/non-retracted full-text dataset. The notebook prepares section-aware document
# representations, trains/evaluates classical and transformer-based models, compares section
# contributions, audits explanation faithfulness, evaluates calibration, tests temporal
# robustness, and freezes dissertation-ready outputs.
#
# This notebook includes:
# - Dataset loading, schema validation, section mapping, and leakage checks
# - Text cleaning with explicit removal of post-publication retraction-status leakage terms
# - Construction of full, core, section-only, and section-masked document views
# - Leakage-safe train/validation/test splitting at group level
# - Classical baselines using TF-IDF + Logistic Regression
# - SciBERT modelling with document-to-chunk tokenisation, Drive caching, and document-level aggregation
# - Longformer long-context comparison across selected document views
# - Section contribution analysis using section-only views and drop-section ablations
# - Explainability using Integrated Gradients, LIME, and SHAP
# - Faithfulness audits using top-k attribution masking and probability-change summaries
# - Calibration analysis using Platt scaling, isotonic regression, ECE, and reliability data
# - Temporal robustness evaluation using past-to-future splits
# - Explanation stability checks over temporal splits
# - Final frozen outputs for dissertation results, appendices, and implementation coverage
#
# Inputs:
# - /content/drive/MyDrive/Dissertation/0. Edinburgh-Napier, Data Engineering/Databases/Step 4. preprocessed_db_final_for_modelling.csv
#   Final paired modelling dataset with article identifiers, labels, metadata, year, and canonical
#   text sections: title, abstract, introduction, methods, results, and discussion.
#
# Main outputs:
# - run_<timestamp>/ folder under Analysis_Final/analysis_artifacts
# - split_manifest.csv
# - qa_missing_sections_raw.json
# - qa_retraction_leakage_after_cleaning.json
# - metrics_tfidf_lr.json
# - results_master.csv
# - final_results_table.csv
# - final_section_ablation_table.csv
# - model_registry.json
# - xai_faithfulness_master.csv
# - Integrated Gradients, LIME, and SHAP explanation exports
# - calibration metrics, ECE metrics, and reliability-curve data
# - temporal robustness and temporal calibration outputs
# - methodology_implementation_check.csv
# - final artefact checklist
#
# Execution:
# - Run top-to-bottom in a fresh Colab session.
# - The notebook is designed to be partly resume-safe through prepared-text and token/chunk caches.
# - Longformer, LIME, SHAP, and temporal sections are heavier; run them only when the preceding
#   SciBERT/checkpoint and prepared-data cells have completed successfully.
#
# Notes:
# This notebook is the final analytical stage of the dissertation pipeline. It is intended to
# produce reproducible, dissertation-ready evidence for model performance, section contribution,
# calibration, temporal robustness, and explanation trustworthiness.

In [ ]:
# Notebook structure:
# - PART A — Cells 1–4: Environment setup, imports, paths, and reproducibility
# - PART B — Cells 5–7: Dataset loading, text preparation, leakage control, and splitting
# - PART C — Cells 8–11: Baseline modelling and transformer configuration
# - PART D — Cells 12–18: SciBERT chunking, training, prediction, and section ablation
# - PART E — Cells 19–24: Integrated Gradients and faithfulness auditing
# - PART F — Cells 25–30: Experiment registry, result consolidation, and Longformer comparison
# - PART G — Cells 31–36: XAI summaries, calibration, ECE, and reliability analysis
# - PART H — Cells 37–40: Temporal split, classical model extensions, and dataset freeze
# - PART I — Cells 41–46: Final dissertation tables, artefact freezing, and XAI reporting
# - PART J — Cells 47–52: LIME, SHAP, extended temporal robustness, and explanation stability
# - PART K — Cells 53–54: Methodology coverage and final artefact checklist

PART A — Environment setup, imports, paths, and reproducibility

In [ ]:
# 1) Mount Google Drive + Install dependencies

from google.colab import drive
drive.mount("/content/drive")

!pip -q install -U transformers datasets accelerate evaluate scikit-learn imbalanced-learn sentencepiece lime
!pip -q install -U "shap>=0.50.0" "captum" --no-cache-dir

In [ ]:
# 2) Imports

import os, re, gc, json, math, time, random, hashlib, warnings, torch
import platform, joblib, shutil, glob, shap, lime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
from datetime import datetime

import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from transformers import AutoConfig
from tqdm.auto import tqdm

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    average_precision_score, roc_auc_score, brier_score_loss,
    precision_recall_curve, roc_curve
)
from sklearn.calibration import calibration_curve
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression as SkLogisticRegression

from lime.lime_text import LimeTextExplainer
from captum.attr import LayerIntegratedGradients

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# 3) Configuration

# Inputs
DB_PATH = "/content/drive/MyDrive/Dissertation/0. Edinburgh-Napier, Data Engineering/Databases/Step 4. preprocessed_db_final_for_modelling.csv"

# Output root
OUT_DIR = "/content/drive/MyDrive/Dissertation/0. Edinburgh-Napier, Data Engineering/Analysis_Final/analysis_artifacts"
os.makedirs(OUT_DIR, exist_ok=True)

RUN_ID = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR = os.path.join(OUT_DIR, f"run_{RUN_ID}")
os.makedirs(RUN_DIR, exist_ok=True)

# Cache root
CACHE_ROOT = "/content/drive/MyDrive/Dissertation/0. Edinburgh-Napier, Data Engineering/cache"
os.makedirs(CACHE_ROOT, exist_ok=True)

# Prepared text cache
TEXT_CACHE_DIR = os.path.join(CACHE_ROOT, "prepared_text")
os.makedirs(TEXT_CACHE_DIR, exist_ok=True)
PREP_CACHE_PATH = os.path.join(TEXT_CACHE_DIR, "prepared_text.parquet")

# Token/chunk cache
TOKEN_CACHE_DIR = os.path.join(CACHE_ROOT, "tokenized_chunks")
os.makedirs(TOKEN_CACHE_DIR, exist_ok=True)

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def save_df(df, path):
    df.to_csv(path, index=False)

print("RUN_DIR:", RUN_DIR)
print("DB_PATH:", DB_PATH)
print("PREP_CACHE_PATH:", PREP_CACHE_PATH)
print("TOKEN_CACHE_DIR:", TOKEN_CACHE_DIR)

In [ ]:
# CELL 4) Schema map + section configuration

# Column map
COLS = {
    "paper_id": "paper_id",
    "doi_norm": "doi_norm",
    "label": "class_label",
    "quality": "quality_percent",
    "applicability": "applicability",

    "title": "section_title",
    "abstract": "section_abstract",
    "introduction": "section_introduction",
    "methods": "section_methods",
    "results": "section_results",
    "discussion": "section_discussion",
}

SECTIONS = ["title", "abstract", "introduction", "methods", "results", "discussion"]

# Document views (these become _doc_<viewname> columns)
# Includes ALL section-only views
DOC_VIEWS = {
    "full": ["title", "abstract", "introduction", "methods", "results", "discussion"],
    "core": ["abstract", "discussion"],

    "title_only": ["title"],
    "abstract_only": ["abstract"],
    "introduction_only": ["introduction"],
    "methods_only": ["methods"],
    "results_only": ["results"],
    "discussion_only": ["discussion"],
}

PART B — Dataset loading, text preparation, leakage control, and splitting

In [ ]:
# CELL 5) Load DB + integrity checks

df = pd.read_csv(DB_PATH)

print("Rows:", len(df), "Cols:", df.shape[1])

# Fail-fast: ensure required columns exist (without modifying them)
required_cols = [COLS[k] for k in ["paper_id", "label"]] + [COLS[s] for s in SECTIONS]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in CSV: {missing}")

# Ensure stable types for keys
df[COLS["paper_id"]] = df[COLS["paper_id"]].astype(str)
if COLS["doi_norm"] in df.columns:
    df[COLS["doi_norm"]] = df[COLS["doi_norm"]].astype(str)

# Basic label sanity
print("Label counts:")
print(df[COLS["label"]].value_counts(dropna=False))

# Create a stable group key for leakage prevention (doi_norm preferred)
def build_group_key(row):
    doi = str(row.get(COLS["doi_norm"], "")).strip().lower()
    pid = str(row.get(COLS["paper_id"], "")).strip()
    if doi and doi != "nan":
        return f"doi:{doi}"
    return f"pid:{pid}"

df["_group_key"] = df.apply(build_group_key, axis=1)

# Light QA: missing section rates (raw, pre-clean)
miss = {}
for s in SECTIONS:
    col = COLS[s]
    miss[s] = float(df[col].isna().mean())
save_json({"missing_rate_by_section_raw": miss}, os.path.join(RUN_DIR, "qa_missing_sections_raw.json"))
print("Saved QA missingness (raw).")

analysis_df = df.copy()

In [ ]:
# CELL 6) Text cleaning + document builders
# Includes targeted removal of post-publication leakage terms

# ==========================
# TEXT PREP + INCREMENTAL CACHE (paper_id-only join)
# ==========================

LEAKAGE_PATTERNS = [
    # Direct retraction-status terms
    r"\bretracted\b",
    r"\bretraction\b",
    r"\bretractions\b",

    # Common publisher / database contamination phrases
    r"\bthis article has been retracted\b",
    r"\bthis paper has been retracted\b",
    r"\bthis publication has been retracted\b",
    r"\bthis manuscript has been retracted\b",
    r"\bretraction notice\b",
    r"\bnotice of retraction\b",
    r"\bretracted article\b",
    r"\bretracted paper\b",
    r"\bretracted publication\b",

    # Common wording around article status
    r"\barticle retracted\b",
    r"\bpaper retracted\b",
    r"\bpublication retracted\b",
    r"\bretracted by\b",
    r"\bretracted from\b",
    r"\bwithdrawn article\b",
    r"\bwithdrawal notice\b",
]

def remove_retraction_leakage(text: str) -> str:
    """
    Remove post-publication leakage terms that directly reveal retraction status.
    This is intentionally conservative: it removes explicit retraction-status
    language but does not remove broader scientific terms or ordinary content.
    """
    if pd.isna(text):
        return ""

    text = str(text)

    for pat in LEAKAGE_PATTERNS:
        text = re.sub(pat, " ", text, flags=re.IGNORECASE)

    return text


def norm_text(x: str) -> str:
    """
    Normalise text and remove explicit post-publication retraction-status leakage.
    """
    if pd.isna(x):
        return ""

    x = str(x)
    x = x.replace("\u00a0", " ")

    # Remove explicit leakage before final whitespace normalisation
    x = remove_retraction_leakage(x)

    x = re.sub(r"\s+", " ", x).strip()
    return x


def make_doc_from_clean_row(row, section_list: List[str]) -> str:
    parts = []

    for s in section_list:
        txt = row.get(f"_clean_{s}", "")
        if txt:
            parts.append(txt)

    return "\n\n".join(parts)


# Use paper_id only as cache join key
KEY_COLS = [COLS["paper_id"]]

# Store doi_norm for reference only
AUX_COLS = [COLS["doi_norm"]] if (COLS["doi_norm"] in analysis_df.columns) else []

CLEAN_COLS = [f"_clean_{s}" for s in SECTIONS]
DOC_COLS = [f"_doc_{k}" for k in DOC_VIEWS.keys()]
PREP_COLS = KEY_COLS + AUX_COLS + CLEAN_COLS + DOC_COLS

# Ensure stable types
analysis_df[COLS["paper_id"]] = analysis_df[COLS["paper_id"]].astype(str)

if AUX_COLS:
    analysis_df[COLS["doi_norm"]] = analysis_df[COLS["doi_norm"]].astype(str)

PREP_VERSION = "v2_remove_retraction_leakage"
PREP_META_PATH = PREP_CACHE_PATH + ".meta.json"

cache_valid = False

if os.path.exists(PREP_CACHE_PATH) and os.path.exists(PREP_META_PATH):
    try:
        with open(PREP_META_PATH, "r", encoding="utf-8") as f:
            prep_meta = json.load(f)

        cache_valid = prep_meta.get("prep_version") == PREP_VERSION

    except Exception:
        cache_valid = False

if cache_valid:
    prep_cached = pd.read_parquet(PREP_CACHE_PATH)
    print(f"Loaded valid prepared-text cache: {PREP_CACHE_PATH}")
else:
    if os.path.exists(PREP_CACHE_PATH):
        print("Existing prepared-text cache found, but it was built with an older preprocessing version.")
        print("Rebuilding prepared-text cache to remove retraction-status leakage.")

    prep_cached = pd.DataFrame(columns=PREP_COLS)


# Ensure columns exist
for c in PREP_COLS:
    if c not in prep_cached.columns:
        prep_cached[c] = "" if c.startswith("_") else np.nan


# Identify rows not yet cached by paper_id
cached_ids = set(prep_cached[COLS["paper_id"]].astype(str).tolist())
need = analysis_df[~analysis_df[COLS["paper_id"]].isin(cached_ids)].copy()

print("Rows in DB:", len(analysis_df))
print("Rows already cached:", len(prep_cached))
print("NEW rows to prepare:", len(need))


if len(need) > 0:
    # Clean section text for NEW rows only
    for s in SECTIONS:
        need[f"_clean_{s}"] = need[COLS[s]].apply(norm_text)

    # Build document views for NEW rows only
    for view_name, sec_list in DOC_VIEWS.items():
        need[f"_doc_{view_name}"] = need.apply(
            lambda r: make_doc_from_clean_row(r, sec_list),
            axis=1
        )

    to_add = need[PREP_COLS].copy()

    # Append + deduplicate by paper_id
    prep_cached = pd.concat([prep_cached, to_add], ignore_index=True)
    prep_cached = prep_cached.drop_duplicates(
        subset=[COLS["paper_id"]],
        keep="last"
    )

    # Save updated cache
    prep_cached.to_parquet(PREP_CACHE_PATH, index=False)

    with open(PREP_META_PATH, "w", encoding="utf-8") as f:
        json.dump(
            {
                "prep_version": PREP_VERSION,
                "leakage_patterns": LEAKAGE_PATTERNS,
                "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            },
            f,
            ensure_ascii=False,
            indent=2
        )

    print("Cache updated:", PREP_CACHE_PATH)
    print("Cache metadata saved:", PREP_META_PATH)


# Merge cached prepared columns back onto the full analysis_df
analysis_df = analysis_df.merge(
    prep_cached[PREP_COLS],
    on=[COLS["paper_id"]],
    how="left"
)

analysis_df["_has_text_full"] = analysis_df["_doc_full"].fillna("").str.len() > 0


# Leakage QA after cleaning
leakage_regex = "|".join(LEAKAGE_PATTERNS)

leakage_qa = {}

for col in DOC_COLS:
    leakage_qa[col] = int(
        analysis_df[col]
        .fillna("")
        .str.contains(leakage_regex, case=False, regex=True)
        .sum()
    )

LEAKAGE_QA_PATH = os.path.join(RUN_DIR, "qa_retraction_leakage_after_cleaning.json")
save_json(leakage_qa, LEAKAGE_QA_PATH)

print("Prepared doc columns:", [c for c in analysis_df.columns if c.startswith("_doc_")][:12], "...")
print("Has any full text:", float(analysis_df["_has_text_full"].mean()))
print("Saved leakage QA:", LEAKAGE_QA_PATH)
print("Leakage hits after cleaning:")
print(leakage_qa)

In [ ]:
# CELL 7) Percent-based split (Train / Val / Test)
# Leakage-safe at group level
# Schema-safe manifest saving

TRAIN_PCT, VAL_PCT, TEST_PCT = 0.70, 0.15, 0.15
assert abs(TRAIN_PCT + VAL_PCT + TEST_PCT - 1.0) < 1e-9

# ---------- label binarization ----------
def to_binary(y):
    if pd.isna(y):
        return None

    # numeric handling
    if isinstance(y, (int, np.integer)):
        return int(y) if y in (0, 1) else None
    if isinstance(y, (float, np.floating)):
        if np.isfinite(y) and y in (0.0, 1.0):
            return int(y)
        return None

    s = str(y).strip().lower()

    if s in {"", "nan", "none", "null"}:
        return None

    # dataset-specific mapping
    if s in {"1", "case", "retracted", "true", "yes", "test"}:
        return 1
    if s in {"0", "control", "nonretracted", "non-retracted", "false", "no"}:
        return 0

    try:
        v = int(s)
        return v if v in (0, 1) else None
    except Exception:
        return None


# ---------- group-level label table ----------
g = (
    analysis_df
    .groupby("_group_key", as_index=False)
    .agg({COLS["label"]: "first"})
    .rename(columns={COLS["label"]: "_label"})
)

g["_y"] = g["_label"].apply(to_binary)

# diagnostics
missing_groups = g[g["_y"].isna()]["_group_key"].astype(str).tolist()
if missing_groups:
    print(f" {len(missing_groups)} groups have missing/unparseable labels and will be DROPPED.")
    print("   Example missing group_keys:", missing_groups[:10])

# keep only valid groups
g = g.dropna(subset=["_y"]).reset_index(drop=True)
g["_y"] = g["_y"].astype(int)

# ---------- push labels back to rows ----------
group_to_y = dict(zip(g["_group_key"], g["_y"]))
mapped = analysis_df["_group_key"].map(group_to_y)

n_bad_rows = int(mapped.isna().sum())
if n_bad_rows > 0:
    print(f"Dropping {n_bad_rows} rows whose group label could not be binarized.")
    analysis_df = analysis_df.loc[~mapped.isna()].copy()
    mapped = mapped.loc[~mapped.isna()]

analysis_df[COLS["label"]] = mapped.astype(int)

# rebuild g from filtered rows to keep alignment
g = (
    analysis_df
    .groupby("_group_key", as_index=False)
    .agg({COLS["label"]: "first"})
    .rename(columns={COLS["label"]: "_y"})
)
g["_y"] = g["_y"].astype(int)

# ---------- stratified group split ----------
sss1 = StratifiedShuffleSplit(
    n_splits=1,
    test_size=(1 - TRAIN_PCT),
    random_state=SEED
)
train_idx, temp_idx = next(sss1.split(g["_group_key"], g["_y"]))
g_train = g.iloc[train_idx].copy()
g_temp  = g.iloc[temp_idx].copy()

val_share_of_temp = VAL_PCT / (VAL_PCT + TEST_PCT)
sss2 = StratifiedShuffleSplit(
    n_splits=1,
    test_size=(1 - val_share_of_temp),
    random_state=SEED
)
val_idx, test_idx = next(sss2.split(g_temp["_group_key"], g_temp["_y"]))
g_val  = g_temp.iloc[val_idx].copy()
g_test = g_temp.iloc[test_idx].copy()

# ---------- propagate split back to rows ----------
split_map = {k: "train" for k in g_train["_group_key"]}
split_map.update({k: "val" for k in g_val["_group_key"]})
split_map.update({k: "test" for k in g_test["_group_key"]})

analysis_df["_split"] = analysis_df["_group_key"].map(split_map)

# ---------- schema-safe manifest saving ----------
manifest_path = os.path.join(RUN_DIR, "split_manifest.csv")

manifest_cols = ["_group_key", COLS["paper_id"], COLS["label"], "_split"]

doi_col = COLS.get("doi_norm")
if doi_col and doi_col in analysis_df.columns:
    manifest_cols.insert(2, doi_col)

missing = [c for c in manifest_cols if c not in analysis_df.columns]
if missing:
    raise ValueError(
        f"Manifest columns missing from analysis_df: {missing}\n"
        f"Available columns (sample): {list(analysis_df.columns)[:40]}"
    )

save_df(analysis_df[manifest_cols], manifest_path)

# ---------- summary ----------
print("Split counts (rows):")
print(analysis_df["_split"].value_counts(dropna=False))

print("\nLabel counts (binary):")
print(analysis_df[COLS["label"]].value_counts(dropna=False))

print("\nSaved:", manifest_path)

PART C — Baseline modelling and transformer configuration

In [ ]:
# CELL 8) TF-IDF baseline (LogReg) for a chosen view

VIEW = "_doc_full"   # change to _doc_abstract_only, _doc_core, etc.

train = analysis_df[analysis_df["_split"]=="train"].copy()
val   = analysis_df[analysis_df["_split"]=="val"].copy()
test  = analysis_df[analysis_df["_split"]=="test"].copy()

X_train, y_train = train[VIEW].fillna(""), train[COLS["label"]].astype(int)
X_val,   y_val   = val[VIEW].fillna(""),   val[COLS["label"]].astype(int)
X_test,  y_test  = test[VIEW].fillna(""),  test[COLS["label"]].astype(int)

tfidf_lr = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1,2),
        min_df=3,
        max_df=0.95,
        strip_accents="unicode",
        lowercase=True
    )),
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        n_jobs=-1
    ))
])

tfidf_lr.fit(X_train, y_train)
val_pred = tfidf_lr.predict_proba(X_val)[:,1]
test_pred = tfidf_lr.predict_proba(X_test)[:,1]

def eval_probs(y_true, p, name="set"):
    out = {}
    out["AUPRC"] = float(average_precision_score(y_true, p))
    out["ROC_AUC"] = float(roc_auc_score(y_true, p))
    out["Brier"] = float(brier_score_loss(y_true, p))
    return out

metrics = {
    "val": eval_probs(y_val, val_pred, "val"),
    "test": eval_probs(y_test, test_pred, "test"),
    "view": VIEW
}
save_json(metrics, os.path.join(RUN_DIR, "metrics_tfidf_lr.json"))
print(metrics)

In [ ]:
# CELL 9) Transformer config + tokenizer (CONFIG ONLY)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

MODEL_NAME = "allenai/scibert_scivocab_uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

# RAM savers:
MAX_LEN = 256
STRIDE = 64
MAX_CHUNKS_PER_DOC = 8

print("MAX_LEN:", MAX_LEN, "| STRIDE:", STRIDE, "| MAX_CHUNKS_PER_DOC:", MAX_CHUNKS_PER_DOC)

In [ ]:
# CELL 10) View-specific max-chunks + cache warmup safety flag

VIEW_MAX_CHUNKS = {
    "_doc_title_only": 2,
    "_doc_abstract_only": 8,
    "_doc_introduction_only": 6,
    "_doc_methods_only": 6,
    "_doc_results_only": 6,
    "_doc_discussion_only": 8,
    "_doc_full": 4,
    "_doc_core": 6,
}

RUN_CACHE_WARMUP = False  # change to True only when you intentionally want to pre-tokenize all views

In [ ]:
# CELL 11) Long-context transformer configuration

LONG_MODEL_NAME = "allenai/longformer-base-4096"

# Long-context comparator.
# 1024 is feasible in Colab and substantially longer than the SciBERT 256-token chunks.
LONG_MAX_LEN = 1024

RUN_LONGFORMER = True

# Run Longformer on several views so it is not only a full-document comparator.
LONGFORMER_TEXT_COLS = [
    "_doc_full",
    "_doc_core",
    "_doc_abstract_only",
    "_doc_methods_only",
    "_doc_results_only",
    "_doc_discussion_only",
]

print("Longformer model:", LONG_MODEL_NAME)
print("LONG_MAX_LEN:", LONG_MAX_LEN)
print("RUN_LONGFORMER:", RUN_LONGFORMER)
print("LONGFORMER_TEXT_COLS:", LONGFORMER_TEXT_COLS)

PART D — SciBERT chunking, training, prediction, and section ablation

In [ ]:
# CELL 12) Chunked dataset (document → multiple tokenized chunks) WITH DRIVE CACHE

# --------------------------
# TOKEN CACHE HELPERS
# --------------------------

def _safe_model_slug(name: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_\-\.]+", "_", name)

MODEL_SLUG = _safe_model_slug(MODEL_NAME)

def text_fingerprint(text: str) -> str:
    if text is None:
        text = ""
    if not isinstance(text, str):
        text = str(text)
    return hashlib.sha1(text.encode("utf-8", errors="ignore")).hexdigest()

def token_cache_paths(paper_id: str, view_name: str):
    """
    Cache location:
      CACHE_ROOT/tokenized_chunks/<model_slug>/<view_name>/<2-char hash>/<paper_id>.npz
      CACHE_ROOT/tokenized_chunks/<model_slug>/<view_name>/<2-char hash>/<paper_id>.json
    """
    h2 = hashlib.sha1(paper_id.encode("utf-8", errors="ignore")).hexdigest()[:2]
    base = os.path.join(CACHE_ROOT, "tokenized_chunks", MODEL_SLUG, view_name, h2)
    os.makedirs(base, exist_ok=True)
    npz_path = os.path.join(base, f"{paper_id}.npz")
    meta_path = os.path.join(base, f"{paper_id}.json")
    return npz_path, meta_path

def load_json(path: str):
    if not os.path.exists(path):
        return None
    with open(path, "r") as f:
        return json.load(f)

def save_json(obj: dict, path: str):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f)
    os.replace(tmp, path)

def load_cached_tokens(paper_id: str, view_name: str):
    npz_path, meta_path = token_cache_paths(paper_id, view_name)
    if not (os.path.exists(npz_path) and os.path.exists(meta_path)):
        return None
    try:
        meta = load_json(meta_path)
        data = np.load(npz_path)
        input_ids = data["input_ids"]
        attention_mask = data["attention_mask"]
        return meta, input_ids, attention_mask
    except Exception:
        return None

def save_cached_tokens(paper_id: str, view_name: str, meta: dict, input_ids: np.ndarray, attention_mask: np.ndarray):
    npz_path, meta_path = token_cache_paths(paper_id, view_name)
    np.savez_compressed(npz_path, input_ids=input_ids, attention_mask=attention_mask)
    save_json(meta, meta_path)  # save_json defined above

def chunk_tokenize_to_arrays(text: str, max_len=256, stride=64, max_chunks=8):
    """
    Tokenize to fixed-size chunks (padded to max_len) and HARD-CAP number of chunks per doc.
    This cap is the key fix that prevents RAM spikes.
    """
    if not text:
        text = ""

    enc = tokenizer(
        text,
        truncation=False,
        padding=False,
        add_special_tokens=True,
        return_attention_mask=True,
        return_tensors=None
    )

    input_ids = enc["input_ids"]
    attn = enc["attention_mask"]

    if len(input_ids) <= max_len:
        # Single chunk: pad to max_len
        ids = input_ids + [tokenizer.pad_token_id] * (max_len - len(input_ids))
        am = attn + [0] * (max_len - len(attn))
        input_ids_arr = np.array([ids], dtype=np.int64)
        attention_mask_arr = np.array([am], dtype=np.int64)
        return input_ids_arr, attention_mask_arr

    # Sliding window
    chunks_ids = []
    chunks_am = []
    step = max_len - stride
    start = 0

    while start < len(input_ids) and len(chunks_ids) < max_chunks:
        end = start + max_len
        ids = input_ids[start:end]
        am = attn[start:end]

        if len(ids) < max_len:
            ids = ids + [tokenizer.pad_token_id] * (max_len - len(ids))
            am = am + [0] * (max_len - len(am))

        chunks_ids.append(ids)
        chunks_am.append(am)
        start += step

    input_ids_arr = np.array(chunks_ids, dtype=np.int64)
    attention_mask_arr = np.array(chunks_am, dtype=np.int64)
    return input_ids_arr, attention_mask_arr


class CachedChunkedDocDataset(Dataset):
    """
    Each item is one document.
    Loads chunk tokens from Drive cache if available; otherwise tokenizes and caches.
    Hard-caps chunks per doc using MAX_CHUNKS_PER_DOC.
    """

    def __init__(
        self,
        df,
        text_col,
        label_col,
        id_col,
        view_name: str,
        max_len=256,
        stride=64,
        max_chunks=8,
        allow_build_cache=True
    ):
        self.df = df.reset_index(drop=True)
        self.text_col = text_col
        self.label_col = label_col
        self.id_col = id_col
        self.view_name = view_name
        self.max_len = max_len
        self.stride = stride
        self.max_chunks = max_chunks
        self.allow_build_cache = allow_build_cache

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        paper_id = str(row[self.id_col])
        y = int(row[self.label_col])

        text = row[self.text_col] if (self.text_col in row and pd.notna(row[self.text_col])) else ""
        fp = text_fingerprint(text)

        # 1) If cache is allowed, try load cached tokens
        if self.allow_build_cache:
            cached = load_cached_tokens(paper_id, self.view_name)
            if cached is not None:
                meta, input_ids, attention_mask = cached

                # Validate cache against current settings
                if (
                    meta.get("text_fp") == fp
                    and meta.get("max_len") == self.max_len
                    and meta.get("stride") == self.stride
                    and meta.get("max_chunks") == self.max_chunks
                    and meta.get("model_name") == MODEL_NAME
                ):
                    return {
                        "paper_id": paper_id,
                        "label": y,
                        "input_ids": input_ids,
                        "attention_mask": attention_mask
                    }

        input_ids, attention_mask = chunk_tokenize_to_arrays(
            text,
            max_len=self.max_len,
            stride=self.stride,
            max_chunks=self.max_chunks
        )

        if self.allow_build_cache:
            meta = {
                "paper_id": paper_id,
                "view_name": self.view_name,
                "model_name": MODEL_NAME,
                "max_len": self.max_len,
                "stride": self.stride,
                "max_chunks": self.max_chunks,
                "text_fp": fp,
                "n_chunks": int(input_ids.shape[0]),
            }
            save_cached_tokens(paper_id, self.view_name, meta, input_ids, attention_mask)

        return {
            "paper_id": paper_id,
            "label": y,
            "input_ids": input_ids,
            "attention_mask": attention_mask
        }


def collate_chunked_docs(batch):
    """
    Collate:
      - flatten chunks across docs into one big batch of chunks
      - keep doc_ids (paper_id) so we can aggregate chunk logits to doc logits later
    """
    input_ids_arr = []
    attention_mask_arr = []
    labels = []
    doc_ids = []

    for item in batch:
        inp = item["input_ids"]
        att = item["attention_mask"]
        y = item["label"]
        pid = item["paper_id"]

        input_ids_arr.append(inp)
        attention_mask_arr.append(att)

        n_chunks = inp.shape[0]
        labels.extend([y] * n_chunks)
        doc_ids.extend([pid] * n_chunks)

    input_ids_arr = np.concatenate(input_ids_arr, axis=0)
    attention_mask_arr = np.concatenate(attention_mask_arr, axis=0)

    return {
        "input_ids": torch.tensor(input_ids_arr, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask_arr, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
        "doc_ids": doc_ids
    }


# Backwards-compatible alias used by later training/evaluation cells
collate_cached_chunked = collate_chunked_docs

In [ ]:
# CELL 13) SAFE token cache warmup for MULTIPLE views (optional, sequential, batched, capped)

def hard_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def warm_cache_for_view(view_col: str, *, limit=None, batch_size=200, max_chunks=None):
    """
    Warm token cache for one view without excessive memory use.
    Forces __getitem__ to run and write cached tokens to Drive.
    """
    view_name = view_col.replace("_doc_", "")
    df = analysis_df

    if limit is not None:
        df = df.head(limit)

    id_col = COLS["paper_id"]
    label_col = COLS["label"]

    _max_chunks = MAX_CHUNKS_PER_DOC if max_chunks is None else max_chunks

    print(f"\n Warming cache for {view_col} | docs={len(df)} | batch={batch_size} | max_chunks={_max_chunks}")
    start = time.time()
    done = 0

    for i in range(0, len(df), batch_size):
        chunk_df = df.iloc[i:i+batch_size]

        ds = CachedChunkedDocDataset(
            chunk_df,
            text_col=view_col,
            label_col=label_col,
            id_col=id_col,
            view_name=view_name,
            max_len=MAX_LEN,
            stride=STRIDE,
            max_chunks=_max_chunks,
            allow_build_cache=True
        )

        for j in range(len(ds)):
            _ = ds[j]

        done += len(ds)

        del ds, chunk_df
        hard_cleanup()

        if done % (batch_size * 5) == 0:
            elapsed = (time.time() - start) / 60
            print(f"  cached {done}/{len(df)} docs | elapsed {elapsed:.1f} min")

    elapsed = (time.time() - start) / 60
    print(f"Done: {view_col} | cached {done} docs | total {elapsed:.1f} min")


VIEWS_TO_WARM = list(VIEW_MAX_CHUNKS.items())
LIMIT = None
BATCH_SIZE = 150

if RUN_CACHE_WARMUP:
    for view_col, max_chunks in VIEWS_TO_WARM:
        if view_col not in analysis_df.columns:
            print(f"Skipping {view_col} (column not found in analysis_df)")
            continue
        warm_cache_for_view(view_col, limit=LIMIT, batch_size=BATCH_SIZE, max_chunks=max_chunks)

    print("\nCache warmup finished.")
else:
    print("Cache warmup skipped. Set RUN_CACHE_WARMUP=True if you want to pre-tokenize all views.")

In [ ]:
# CELL 14) Model + training loop with visible progress

USE_AMP = (DEVICE == "cuda")
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


def _cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def train_one_epoch(model, loader, optim, scheduler=None, grad_accum_steps=1):
    model.train()
    total_loss = 0.0
    optim.zero_grad(set_to_none=True)

    pbar = tqdm(
        enumerate(loader, start=1),
        total=len(loader),
        desc="Training batches",
        leave=True
    )

    for step, batch in pbar:
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attn = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            out = model(
                input_ids=input_ids,
                attention_mask=attn,
                labels=labels
            )
            loss = out.loss / grad_accum_steps

        scaler.scale(loss).backward()

        if step % grad_accum_steps == 0:
            scaler.unscale_(optim)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optim)
            scaler.update()
            optim.zero_grad(set_to_none=True)

            if scheduler:
                scheduler.step()

        batch_loss = float(loss.item()) * grad_accum_steps
        total_loss += batch_loss

        if step == 1 or step % 25 == 0:
            pbar.set_postfix({
                "loss": f"{batch_loss:.4f}",
                "step": f"{step}/{len(loader)}"
            })

    # Flush remaining gradients if number of batches is not divisible by grad_accum_steps
    if (len(loader) % grad_accum_steps) != 0:
        scaler.unscale_(optim)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optim)
        scaler.update()
        optim.zero_grad(set_to_none=True)

        if scheduler:
            scheduler.step()

    return total_loss / max(1, len(loader))


@torch.no_grad()
def predict_doc_probs(model, loader, agg="mean"):
    model.eval()
    doc2probs, doc2labels = {}, {}

    pbar = tqdm(
        loader,
        total=len(loader),
        desc=f"Predicting docs ({agg})",
        leave=False
    )

    for batch in pbar:
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attn = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].cpu().numpy()
        doc_ids = batch["doc_ids"]

        logits = model(
            input_ids=input_ids,
            attention_mask=attn
        ).logits

        probs = F.softmax(logits, dim=-1)[:, 1].detach().cpu().numpy()

        for p, y, d in zip(probs, labels, doc_ids):
            doc2labels[d] = int(y)
            doc2probs.setdefault(d, []).append(float(p))

    doc_ids_sorted = sorted(doc2probs.keys())
    y_true = np.array([doc2labels[d] for d in doc_ids_sorted], dtype=int)

    if agg == "max":
        y_prob = np.array(
            [max(doc2probs[d]) for d in doc_ids_sorted],
            dtype=float
        )
    else:
        y_prob = np.array(
            [np.mean(doc2probs[d]) for d in doc_ids_sorted],
            dtype=float
        )

    return doc_ids_sorted, y_true, y_prob


def run_transformer_experiment(
    text_view_col: str,
    epochs=1,
    batch_docs=1,
    lr=2e-5,
    agg="mean",
    grad_accum_steps=16,
    max_len=None,
    stride=None,
    max_chunks=None,
):
    train_df = analysis_df[analysis_df["_split"] == "train"].copy()
    val_df   = analysis_df[analysis_df["_split"] == "val"].copy()
    test_df  = analysis_df[analysis_df["_split"] == "test"].copy()

    id_col = COLS["paper_id"]
    label_col = COLS["label"]
    view_name = text_view_col.replace("_doc_", "")

    _max_len = MAX_LEN if max_len is None else max_len
    _stride = STRIDE if stride is None else stride
    _max_chunks = MAX_CHUNKS_PER_DOC if max_chunks is None else max_chunks

    print("\n=== Transformer experiment ===")
    print("View:", text_view_col)
    print("Train docs:", len(train_df))
    print("Val docs:", len(val_df))
    print("Test docs:", len(test_df))
    print("max_len:", _max_len)
    print("stride:", _stride)
    print("max_chunks:", _max_chunks)
    print("batch_docs:", batch_docs)
    print("grad_accum_steps:", grad_accum_steps)
    print("device:", DEVICE)

    ds_train = CachedChunkedDocDataset(
        train_df,
        text_view_col,
        label_col,
        id_col,
        view_name=view_name,
        max_len=_max_len,
        stride=_stride,
        max_chunks=_max_chunks,
        allow_build_cache=True
    )

    ds_val = CachedChunkedDocDataset(
        val_df,
        text_view_col,
        label_col,
        id_col,
        view_name=view_name,
        max_len=_max_len,
        stride=_stride,
        max_chunks=_max_chunks,
        allow_build_cache=True
    )

    ds_test = CachedChunkedDocDataset(
        test_df,
        text_view_col,
        label_col,
        id_col,
        view_name=view_name,
        max_len=_max_len,
        stride=_stride,
        max_chunks=_max_chunks,
        allow_build_cache=True
    )

    dl_train = DataLoader(
        ds_train,
        batch_size=batch_docs,
        shuffle=True,
        collate_fn=collate_cached_chunked,
        num_workers=0,
        pin_memory=(DEVICE == "cuda")
    )

    dl_val = DataLoader(
        ds_val,
        batch_size=batch_docs,
        shuffle=False,
        collate_fn=collate_cached_chunked,
        num_workers=0,
        pin_memory=(DEVICE == "cuda")
    )

    dl_test = DataLoader(
        ds_test,
        batch_size=batch_docs,
        shuffle=False,
        collate_fn=collate_cached_chunked,
        num_workers=0,
        pin_memory=(DEVICE == "cuda")
    )

    print("Train batches:", len(dl_train))
    print("Val batches:", len(dl_val))
    print("Test batches:", len(dl_test))

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    ).to(DEVICE)

    optim = torch.optim.AdamW(model.parameters(), lr=lr)

    num_steps = epochs * max(1, len(dl_train))

    scheduler = get_linear_schedule_with_warmup(
        optim,
        num_warmup_steps=max(10, int(0.05 * num_steps)),
        num_training_steps=num_steps
    )

    history = {
        "meta": {
            "text_view_col": text_view_col,
            "view_name": view_name,
            "epochs": epochs,
            "batch_docs": batch_docs,
            "lr": lr,
            "agg": agg,
            "grad_accum_steps": grad_accum_steps,
            "max_len": _max_len,
            "stride": _stride,
            "max_chunks": _max_chunks,
            "model_name": MODEL_NAME,
        },
        "train_loss": [],
        "val": [],
        "test": [],
        "ckpt_dirs": []
    }

    for ep in range(1, epochs + 1):
        print(f"\n--- Epoch {ep}/{epochs} ---")

        loss = train_one_epoch(
            model,
            dl_train,
            optim,
            scheduler,
            grad_accum_steps=grad_accum_steps
        )

        history["train_loss"].append(float(loss))

        _, yv, pv = predict_doc_probs(model, dl_val, agg=agg)
        val_metrics = eval_probs(yv, pv, "val")
        history["val"].append(val_metrics)

        _, yt, pt = predict_doc_probs(model, dl_test, agg=agg)
        test_metrics = eval_probs(yt, pt, "test")
        history["test"].append(test_metrics)

        print(
            f"Epoch {ep}/{epochs} | "
            f"loss={loss:.4f} | "
            f"val={val_metrics} | "
            f"test={test_metrics}"
        )

        ckpt_dir = os.path.join(RUN_DIR, f"ckpt_{view_name}_ep{ep}")
        os.makedirs(ckpt_dir, exist_ok=True)

        model.save_pretrained(ckpt_dir)
        tokenizer.save_pretrained(ckpt_dir)

        history["ckpt_dirs"].append(ckpt_dir)

        save_json(
            history,
            os.path.join(RUN_DIR, f"history_transformer_{view_name}.json")
        )

        print("Saved checkpoint:", ckpt_dir)

        _cleanup_cuda()

    del ds_train, ds_val, ds_test, dl_train, dl_val, dl_test
    _cleanup_cuda()

    return model, history

In [ ]:
# CELL 15) Helpers: build masked document views (by section)

# --------------------------
# SECTION MASKING / ABLATION (FULL DOC VARIANTS)
# --------------------------

def build_masked_doc_from_clean(row, drop_sections, base_view_sections):
    parts = []
    for s in base_view_sections:
        if s in drop_sections:
            continue
        txt = row.get(f"_clean_{s}", "")
        if txt:
            parts.append(txt)
    return "\n\n".join(parts)

# Base full document sections in order
BASE_FULL_SECS = ["title", "abstract", "introduction", "methods", "results", "discussion"]

# Ensure required clean columns exist
for s in BASE_FULL_SECS:
    col = f"_clean_{s}"
    if col not in analysis_df.columns:
        raise ValueError(f"Missing '{col}'. Run Cell 6 (prepared text + cache merge) first.")

# Mask specs: drop each section individually + a couple of useful combos
MASK_SPECS = {
    "drop_title": ["title"],
    "drop_abstract": ["abstract"],
    "drop_introduction": ["introduction"],
    "drop_methods": ["methods"],
    "drop_results": ["results"],
    "drop_discussion": ["discussion"],

    # Optional combos (keep small and interpretable)
    "drop_abstract_discussion": ["abstract", "discussion"],
    "drop_methods_results": ["methods", "results"],
}

# Create masked full-doc columns
for name, drop_secs in MASK_SPECS.items():
    col = f"_doc_full__{name}"
    analysis_df[col] = analysis_df.apply(
        lambda r: build_masked_doc_from_clean(r, drop_secs, BASE_FULL_SECS),
        axis=1
    )

print("Masked doc columns created:")
print([f"_doc_full__{k}" for k in MASK_SPECS.keys()])

In [ ]:
# CELL 16) Ensure checkpoint exists (auto-train if none), then load latest checkpoint

def is_hf_checkpoint_dir(path: str) -> bool:
    if not path or not os.path.isdir(path):
        return False
    files = set(os.listdir(path))
    has_config = "config.json" in files
    has_weights = any(f in files for f in ["pytorch_model.bin", "model.safetensors"])
    return has_config and has_weights

def find_latest_ckpt(run_dir: str, prefix="ckpt_", contains=None):
    cands = []
    for name in os.listdir(run_dir):
        p = os.path.join(run_dir, name)
        if not os.path.isdir(p):
            continue
        if not name.startswith(prefix):
            continue
        if contains and contains not in name:
            continue
        if not is_hf_checkpoint_dir(p):
            continue
        cands.append(p)
    if not cands:
        return None
    cands.sort(key=lambda p: os.path.getmtime(p), reverse=True)
    return cands[0]

def _cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# 1) Try to find an existing valid HF checkpoint
ckpt = find_latest_ckpt(RUN_DIR, prefix="ckpt_", contains=None)

# 2) If none found, train the main SciBERT checkpoint required by downstream analyses
if ckpt is None:
    print("No VALID checkpoint found in this run folder. Training the main SciBERT checkpoint now...")

    BASE_TRAIN_VIEW = "_doc_full"

    _model_tmp, _hist_tmp = run_transformer_experiment(
        text_view_col=BASE_TRAIN_VIEW,
        epochs=1,
        batch_docs=1,
        lr=2e-5,
        agg="mean",
        grad_accum_steps=16,
        max_len=MAX_LEN,
        stride=STRIDE,
        max_chunks=VIEW_MAX_CHUNKS.get(BASE_TRAIN_VIEW, MAX_CHUNKS_PER_DOC)
    )

    # cleanup immediately
    del _model_tmp, _hist_tmp
    _cleanup()

    ckpt = find_latest_ckpt(RUN_DIR, prefix="ckpt_", contains=BASE_TRAIN_VIEW.replace("_doc_", ""))
    if ckpt is None:
        ckpt = find_latest_ckpt(RUN_DIR, prefix="ckpt_", contains=None)

# 3) Final guard
if ckpt is None or not is_hf_checkpoint_dir(ckpt):
    raise FileNotFoundError(
        f"Could not locate a loadable HuggingFace checkpoint under RUN_DIR={RUN_DIR}. "
        f"Expected a folder containing config.json and (pytorch_model.bin or model.safetensors)."
    )

CKPT_DIR = ckpt
tokenizer = AutoTokenizer.from_pretrained(CKPT_DIR, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(CKPT_DIR).to(DEVICE)

print("Loaded checkpoint:", CKPT_DIR)
print("Device:", DEVICE)

_cleanup()

In [ ]:
# CELL 17) Get probabilities and metrics for any text column

CACHE_MASKED = False  # Masked views are generated for ablation and are usually not cached to avoid unnecessary Drive usage

@torch.no_grad()
def get_doc_probs_for_col(df_subset, text_col, agg="mean", batch_docs=2):
    view_name = text_col.replace("_doc_", "")

    allow_cache = True
    if (not CACHE_MASKED) and text_col.startswith("_doc_full__"):
        allow_cache = False

    max_chunks = VIEW_MAX_CHUNKS.get(text_col, MAX_CHUNKS_PER_DOC)

    ds = CachedChunkedDocDataset(
        df_subset,
        text_col,
        COLS["label"],
        COLS["paper_id"],
        view_name=view_name,
        max_len=MAX_LEN,
        stride=STRIDE,
        max_chunks=max_chunks,
        allow_build_cache=allow_cache
    )

    dl = DataLoader(
        ds,
        batch_size=batch_docs,
        shuffle=False,
        collate_fn=collate_cached_chunked
    )

    doc_ids, y_true, y_prob = predict_doc_probs(model, dl, agg=agg)
    return doc_ids, y_true, y_prob


@torch.no_grad()
def eval_transformer_on_col(df_subset, text_col, agg="mean", batch_docs=2):
    _, y_true, y_prob = get_doc_probs_for_col(
        df_subset,
        text_col,
        agg=agg,
        batch_docs=batch_docs
    )

    return eval_probs(y_true, y_prob, name=text_col), y_true, y_prob


train_df = analysis_df[analysis_df["_split"] == "train"].copy()
val_df   = analysis_df[analysis_df["_split"] == "val"].copy()
test_df  = analysis_df[analysis_df["_split"] == "test"].copy()

base_metrics_test, base_y, base_p = eval_transformer_on_col(
    test_df,
    "_doc_full",
    agg="mean",
    batch_docs=2
)

print("BASE test metrics:", base_metrics_test)
save_json({"base_test": base_metrics_test}, os.path.join(RUN_DIR, "masking_base_metrics.json"))

In [ ]:
# CELL 18) Section masking / ablation: run all masks + compute deltas

results = []

base = base_metrics_test

for spec_name in MASK_SPECS.keys():
    masked_col = f"_doc_full__{spec_name}"
    m_test, _, _ = eval_transformer_on_col(test_df, masked_col, agg="mean", batch_docs=2)

    row = {
        "mask": spec_name,
        "col": masked_col,
        "test_AUPRC": m_test["AUPRC"],
        "test_ROC_AUC": m_test["ROC_AUC"],
        "test_Brier": m_test["Brier"],
        "delta_AUPRC": m_test["AUPRC"] - base["AUPRC"],
        "delta_ROC_AUC": m_test["ROC_AUC"] - base["ROC_AUC"],
        "delta_Brier": m_test["Brier"] - base["Brier"],  # positive usually worse (higher Brier)
    }
    results.append(row)
    print(row)

mask_df = pd.DataFrame(results).sort_values("delta_AUPRC")
mask_path = os.path.join(RUN_DIR, "section_masking_results.csv")
mask_df.to_csv(mask_path, index=False)

print("Saved:", mask_path)
mask_df.head(10)

PART E — Integrated Gradients and faithfulness auditing

In [ ]:
# CELL 19) Integrated Gradients setup with Captum

def _get_embedding_layer(m):
    """
    SciBERT is BERT-like. This returns the embedding module robustly.
    """
    if hasattr(m, "bert") and hasattr(m.bert, "embeddings"):
        return m.bert.embeddings
    if hasattr(m, "roberta") and hasattr(m.roberta, "embeddings"):
        return m.roberta.embeddings
    if hasattr(m, "base_model") and hasattr(m.base_model, "embeddings"):
        return m.base_model.embeddings
    for attr in ["embeddings"]:
        if hasattr(m, attr):
            return getattr(m, attr)
    raise ValueError("Could not locate embeddings layer for LayerIntegratedGradients.")

EMB_LAYER = _get_embedding_layer(model)

def forward_for_lig(input_ids, attention_mask):
    """
    Return probability for class 1 so attributions align with the positive/retracted class.
    """
    out = model(input_ids=input_ids, attention_mask=attention_mask)
    probs = F.softmax(out.logits, dim=-1)[:, 1]
    return probs

lig = LayerIntegratedGradients(forward_for_lig, EMB_LAYER)

print("Captum LayerIntegratedGradients ready.")

In [ ]:
# CELL 20) Token attribution extraction (single chunk; aligned encoding)

def encode_single_chunk(text: str, max_len=MAX_LEN):
    enc = tokenizer(
        text if text else "",
        truncation=True,
        max_length=max_len,
        return_tensors="pt",
        padding="max_length"
    )
    return enc["input_ids"].to(DEVICE), enc["attention_mask"].to(DEVICE)

def get_token_attributions(text: str, max_len=MAX_LEN, n_steps=24):
    """
    Returns:
      tokens: list[str]
      scores: np.array shape (seq_len,)
      input_ids, attention_mask: tensors
    """
    input_ids, attention_mask = encode_single_chunk(text, max_len=max_len)

    # Baseline: PAD tokens (common and defensible baseline for BERT-like models)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
    baseline_ids = torch.full_like(input_ids, fill_value=pad_id)

    # Attribute with LayerIntegratedGradients on embeddings
    attributions = lig.attribute(
        inputs=input_ids,
        baselines=baseline_ids,
        additional_forward_args=(attention_mask,),
        n_steps=n_steps
    )  # shape: (1, seq_len, hidden_dim)

    # Reduce over embedding dim -> per-token score
    token_scores = attributions.sum(dim=-1).squeeze(0).detach().cpu().numpy()

    # Convert ids to tokens
    tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze(0).detach().cpu().tolist())

    return tokens, token_scores, input_ids, attention_mask

In [ ]:
# CELL 21) Faithfulness masking (mask top-k tokens by attribution, same encoding)

@torch.no_grad()
def prob_case_from_encoding(input_ids, attention_mask):
    out = model(input_ids=input_ids, attention_mask=attention_mask)
    return float(F.softmax(out.logits, dim=-1)[0, 1].detach().cpu().item())

def topk_mask_positions(tokens, scores, k_frac=0.10):
    """
    Choose top-k positions excluding special and pad tokens.
    """
    seq_len = len(tokens)
    k = max(1, int(seq_len * k_frac))

    special = set(tokenizer.all_special_tokens)
    pad_tok = tokenizer.pad_token if tokenizer.pad_token else "[PAD]"

    candidates = []
    for i, tok in enumerate(tokens):
        if tok in special or tok == pad_tok:
            continue
        candidates.append(i)

    if not candidates:
        return []

    # rank candidates by absolute attribution (stronger faithfulness test)
    ranked = sorted(candidates, key=lambda i: abs(scores[i]), reverse=True)
    return ranked[:k]

def mask_positions(input_ids, positions):
    masked = input_ids.clone()
    mask_id = tokenizer.mask_token_id
    if mask_id is None:
        raise ValueError("Tokenizer has no mask_token_id; cannot run masking faithfulness test.")
    for i in positions:
        masked[0, i] = mask_id
    return masked

def faithfulness_test(text: str, k_fracs=(0.05, 0.10, 0.20), n_steps=24):
    tokens, scores, input_ids, attention_mask = get_token_attributions(text, max_len=MAX_LEN, n_steps=n_steps)
    p0 = prob_case_from_encoding(input_ids, attention_mask)

    out = {"p0": p0, "k_fracs": {}, "tokens": tokens}

    for kf in k_fracs:
        pos = topk_mask_positions(tokens, scores, k_frac=kf)
        masked_ids = mask_positions(input_ids, pos)
        p_mask = prob_case_from_encoding(masked_ids, attention_mask)
        out["k_fracs"][str(kf)] = {
            "k": len(pos),
            "p_mask": p_mask,
            "delta": p_mask - p0
        }

    return out

In [ ]:
# CELL 22) Global experiment config + utility writers

EXP_NAME = "scibert_baseline_runner"
EXP_DIR = os.path.join(RUN_DIR, EXP_NAME)
os.makedirs(EXP_DIR, exist_ok=True)

XAI_DIR = os.path.join(EXP_DIR, "xai")
os.makedirs(XAI_DIR, exist_ok=True)

RESULTS_PATH = os.path.join(EXP_DIR, "results_master.csv")
XAI_AUDIT_PATH = os.path.join(XAI_DIR, "xai_faithfulness_master.csv")

def now_str():
    return datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")

def append_csv_row(path, row_dict, header_order=None):
    df_row = pd.DataFrame([row_dict])
    if header_order:
        df_row = df_row.reindex(columns=header_order)

    if os.path.exists(path):
        df_row.to_csv(path, mode="a", header=False, index=False)
    else:
        df_row.to_csv(path, mode="w", header=True, index=False)

print("EXP_DIR:", EXP_DIR)
print("RESULTS_PATH:", RESULTS_PATH)
print("XAI_AUDIT_PATH:", XAI_AUDIT_PATH)

In [ ]:
# CELL 23) Integrated Gradients faithfulness example + small-sample summary

# Ensure EXP_DIR exists even if Cell 22 hasn't run yet
if "EXP_DIR" not in globals():
    EXP_NAME = "scibert_baseline_runner"
    EXP_DIR = os.path.join(RUN_DIR, EXP_NAME)
    os.makedirs(EXP_DIR, exist_ok=True)

XAI_DIR = os.path.join(EXP_DIR, "xai")
os.makedirs(XAI_DIR, exist_ok=True)

# Pick one example from test set (prefer a positive if available)
test_df = analysis_df[analysis_df["_split"]=="test"].copy()

if len(test_df) == 0:
    raise ValueError("No test rows found. Check your split in Cell 6.")

pos = test_df[test_df[COLS["label"]] == 1]
row = (pos.sample(1, random_state=SEED).iloc[0] if len(pos) > 0 else test_df.sample(1, random_state=SEED).iloc[0])

paper_id = str(row[COLS["paper_id"]])

# Prefer abstract-only for XAI (clean + interpretable); fallback to full
text_col = "_doc_abstract_only" if "_doc_abstract_only" in test_df.columns else "_doc_full"
text_example = row[text_col] if pd.notna(row[text_col]) else ""

if not str(text_example).strip():
    raise ValueError(f"Selected example (paper_id={paper_id}) has empty text in {text_col}. "
                     f"Pick another example or ensure you built doc views in Cell 6.")

example = faithfulness_test(text_example, k_fracs=(0.05, 0.10, 0.20), n_steps=24)
example["paper_id"] = paper_id
example["text_col"] = text_col
save_json(example, os.path.join(XAI_DIR, "xai_faithfulness_single_example.json"))

print("Saved single XAI example:", os.path.join(XAI_DIR, "xai_faithfulness_single_example.json"))

# Small aggregate summary (sample only; keeps compute manageable)
SAMPLE_N = min(50, len(test_df))
sample_rows = test_df.sample(SAMPLE_N, random_state=SEED)

deltas = {0.05: [], 0.10: [], 0.20: []}

for _, r in sample_rows.iterrows():
    txt = r[text_col] if pd.notna(r[text_col]) else ""
    if not str(txt).strip():
        continue

    out = faithfulness_test(txt, k_fracs=(0.05, 0.10, 0.20), n_steps=12)  # fewer steps for speed
    for kf in [0.05, 0.10, 0.20]:
        deltas[kf].append(out["k_fracs"][str(kf)]["delta"])

summary = {
    "sample_n_requested": int(SAMPLE_N),
    "sample_n_used_nonempty": int(min(len(sample_rows), max(len(deltas[0.05]), len(deltas[0.10]), len(deltas[0.20])))),
    "text_col": text_col,
    "mean_delta": {str(k): float(np.mean(v)) if len(v) else np.nan for k, v in deltas.items()},
    "median_delta": {str(k): float(np.median(v)) if len(v) else np.nan for k, v in deltas.items()},
}

save_json(summary, os.path.join(XAI_DIR, "xai_faithfulness_summary.json"))
print("Saved XAI summary:", os.path.join(XAI_DIR, "xai_faithfulness_summary.json"))
print(summary)

In [ ]:
# CELL 24) Batch faithfulness audit (sample N papers, report average degradation)

def faithfulness_audit(df_subset, text_col, n=40, k_fracs=(0.05, 0.10, 0.20), n_steps=12, seed=SEED):
    samp = df_subset.sample(min(n, len(df_subset)), random_state=seed).copy()
    rows = []

    for _, r in samp.iterrows():
        text = r[text_col] if pd.notna(r[text_col]) else ""
        if not str(text).strip():
            continue

        out = faithfulness_test(text, k_fracs=k_fracs, n_steps=n_steps)

        row = {
            "paper_id": str(r[COLS["paper_id"]]),
            "text_col": text_col,
            "p0": float(out["p0"]),
        }

        for kf in k_fracs:
            d = out["k_fracs"][str(kf)]
            row[f"k_{kf}"] = int(d["k"])
            row[f"p_mask_{kf}"] = float(d["p_mask"])
            row[f"delta_{kf}"] = float(d["delta"])

        rows.append(row)

    return pd.DataFrame(rows)

# Choose the audit column (abstract-only recommended; consistent with your XAI cell)
AUDIT_TEXT_COL = "_doc_abstract_only" if "_doc_abstract_only" in analysis_df.columns else "_doc_full"

# Build test_df (if not already available in the runtime)
if "test_df" not in globals():
    test_df = analysis_df[analysis_df["_split"] == "test"].copy()

audit_df = faithfulness_audit(
    test_df,
    AUDIT_TEXT_COL,
    n=40,
    k_fracs=(0.05, 0.10, 0.20),
    n_steps=12,
    seed=SEED
)

# Ensure XAI_AUDIT_PATH exists
if "XAI_AUDIT_PATH" not in globals():
    # Fallback: keep everything under RUN_DIR if Cell 22 wasn't run yet
    EXP_NAME = "scibert_baseline_runner"
    EXP_DIR = os.path.join(RUN_DIR, EXP_NAME)
    os.makedirs(EXP_DIR, exist_ok=True)
    XAI_DIR = os.path.join(EXP_DIR, "xai")
    os.makedirs(XAI_DIR, exist_ok=True)
    XAI_AUDIT_PATH = os.path.join(XAI_DIR, "xai_faithfulness_master.csv")

# Append safely (or create)
if os.path.exists(XAI_AUDIT_PATH):
    audit_df.to_csv(XAI_AUDIT_PATH, mode="a", header=False, index=False)
else:
    audit_df.to_csv(XAI_AUDIT_PATH, index=False)

print("Saved XAI batch audit to:", XAI_AUDIT_PATH)

if len(audit_df) == 0:
    print("No non-empty texts were available in the sampled rows.")
else:
    for kf in [0.05, 0.10, 0.20]:
        print(f"Mean ΔP @ {kf}: {audit_df[f'delta_{kf}'].mean():.4f} | Median: {audit_df[f'delta_{kf}'].median():.4f}")

PART F — Experiment registry, result consolidation, and Longformer comparison

In [ ]:
# CELL 25) Evaluate function (single column, val+test, with doc aggregation)

def evaluate_col_on_splits(text_col, agg="mean", batch_docs=2):
    out = {"text_col": text_col, "agg": agg}

    m_val, yv, pv = eval_transformer_on_col(val_df, text_col, agg=agg, batch_docs=batch_docs)
    m_test, yt, pt = eval_transformer_on_col(test_df, text_col, agg=agg, batch_docs=batch_docs)

    out.update({
        "val_AUPRC": m_val["AUPRC"],
        "val_ROC_AUC": m_val["ROC_AUC"],
        "val_Brier": m_val["Brier"],
        "test_AUPRC": m_test["AUPRC"],
        "test_ROC_AUC": m_test["ROC_AUC"],
        "test_Brier": m_test["Brier"],
        "n_val_docs": int(len(np.unique(val_df[COLS["paper_id"]]))),
        "n_test_docs": int(len(np.unique(test_df[COLS["paper_id"]]))),
    })
    return out

In [ ]:
# CELL 26) Model Registry (run-level model specifications)

def build_model_registry():
    cfg = AutoConfig.from_pretrained(MODEL_NAME)

    registry = {
        "run_id": os.path.basename(RUN_DIR),
        "timestamp_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),

        # --- Model ---
        "base_model": MODEL_NAME,
        "architecture": cfg.model_type,
        "num_layers": cfg.num_hidden_layers,
        "hidden_size": cfg.hidden_size,
        "num_attention_heads": cfg.num_attention_heads,
        "num_labels": cfg.num_labels,

        # --- Tokenization & chunking ---
        "tokenization": {
            "tokenizer": tokenizer.__class__.__name__,
            "max_length": MAX_LEN,
            "stride": STRIDE,
            "max_chunks_per_document": MAX_CHUNKS_PER_DOC,
            "vocab_size": cfg.vocab_size,
        },

        # --- Document representations ---
        "document_views": {
            k: v for k, v in DOC_VIEWS.items()
        },

        # --- Aggregation ---
        "aggregation_strategies": ["mean", "max"],

        # --- Training configuration ---
        "training": {
            "optimizer": "AdamW",
            "learning_rate": LR if "LR" in globals() else None,
            "epochs": EPOCHS if "EPOCHS" in globals() else None,
            "batch_documents": BATCH_DOCS if "BATCH_DOCS" in globals() else None,
            "gradient_accumulation_steps": GRAD_ACCUM_STEPS if "GRAD_ACCUM_STEPS" in globals() else None,
            "mixed_precision": torch.cuda.is_available(),
        },

        # --- Explainability ---
        "explainability": {
            "methods": [
                "LayerIntegratedGradients",
                "SHAP"
            ],
            "attribution_target": "P(class=1)",
            "faithfulness_tests": {
                "perturbation_fractions": [0.05, 0.10, 0.20],
                "metric": "Δ probability after top-k token removal"
            }
        },

        # --- Runtime ---
        "runtime": {
            "device": str(DEVICE),
            "torch_version": torch.__version__,
            "numpy_version": np.__version__,
            "python_version": platform.python_version(),
            "platform": platform.platform(),
        }
    }

    return registry


MODEL_REGISTRY = build_model_registry()

registry_path = os.path.join(RUN_DIR, "model_registry.json")
with open(registry_path, "w") as f:
    json.dump(MODEL_REGISTRY, f, indent=2)

print(f"Model registry saved to: {registry_path}")

In [ ]:
# CELL 27) Define the run plan (views + masks + aggregation choices)

# Document views to test
VIEW_COLS = {
    "full": "_doc_full",
    "core": "_doc_core",

    "title_only": "_doc_title_only",
    "abstract_only": "_doc_abstract_only",
    "introduction_only": "_doc_introduction_only",
    "methods_only": "_doc_methods_only",
    "results_only": "_doc_results_only",
    "discussion_only": "_doc_discussion_only",
}

# Ablations already created
MASK_COLS = {k: f"_doc_full__{k}" for k in MASK_SPECS.keys()}

AGGS = ["mean", "max"]  # mean is your main report; max is optional

RUN_PLAN = []

# Baseline views (includes ALL section-only)
for vname, vcol in VIEW_COLS.items():
    for agg in AGGS:
        RUN_PLAN.append(("view", vname, vcol, agg))

# Ablations
for mname, mcol in MASK_COLS.items():
    for agg in AGGS:
        RUN_PLAN.append(("mask", mname, mcol, agg))

print("Planned experiments:", len(RUN_PLAN))
RUN_PLAN[:10]

In [ ]:
# CELL 28) Run all experiments and save one master CSV (+ deltas vs full baseline)

# First compute baseline FULL for each agg (for deltas)
baseline_by_agg = {}
for agg in AGGS:
    baseline_by_agg[agg] = evaluate_col_on_splits("_doc_full", agg=agg, batch_docs=2)

save_json(baseline_by_agg, os.path.join(EXP_DIR, "baseline_full_by_agg.json"))

# Canonical output column order
COL_ORDER = [
    "run_ts", "exp_name", "model_name", "ckpt_dir",
    "kind", "name", "text_col", "agg",
    "val_AUPRC", "val_ROC_AUC", "val_Brier",
    "test_AUPRC", "test_ROC_AUC", "test_Brier",
    "delta_val_AUPRC_vs_full", "delta_test_AUPRC_vs_full",
    "delta_val_Brier_vs_full", "delta_test_Brier_vs_full",
    "n_val_docs", "n_test_docs",
]

MODEL_META = {
    "exp_name": EXP_NAME,
    "model_name": MODEL_NAME,
    "ckpt_dir": CKPT_DIR
}

for kind, name, col, agg in RUN_PLAN:
    metrics = evaluate_col_on_splits(col, agg=agg, batch_docs=2)

    base = baseline_by_agg[agg]
    row = {
        "run_ts": now_str(),
        **MODEL_META,
        "kind": kind,
        "name": name,
        **metrics,
        "delta_val_AUPRC_vs_full": metrics["val_AUPRC"] - base["val_AUPRC"],
        "delta_test_AUPRC_vs_full": metrics["test_AUPRC"] - base["test_AUPRC"],
        "delta_val_Brier_vs_full": metrics["val_Brier"] - base["val_Brier"],
        "delta_test_Brier_vs_full": metrics["test_Brier"] - base["test_Brier"],
    }

    append_csv_row(RESULTS_PATH, row, header_order=COL_ORDER)
    print(f"{kind}:{name} agg={agg} | test_AUPRC={metrics['test_AUPRC']:.4f} Δ={row['delta_test_AUPRC_vs_full']:+.4f}")

print("\nSaved master results to:", RESULTS_PATH)

In [ ]:
# CELL 29) Inspect preliminary SciBERT section/view leaderboard

res = pd.read_csv(RESULTS_PATH)

print("=== Best overall (by test AUPRC) ===")
display(res.sort_values("test_AUPRC", ascending=False).head(10))

print("\n=== Biggest drops vs FULL (masking impact; mean agg only) ===")
mask_mean = res[(res["kind"]=="mask") & (res["agg"]=="mean")].sort_values("delta_test_AUPRC_vs_full")
display(mask_mean.head(10))

print("\n=== Best views (mean agg only) ===")
view_mean = res[(res["kind"]=="view") & (res["agg"]=="mean")].sort_values("test_AUPRC", ascending=False)
display(view_mean)

In [ ]:
# CELL 30) Long-context transformer experiments: Longformer across document views

class LongTextDataset(Dataset):
    """
    One document = one long-tokenized sequence.
    Used for Longformer as the long-context model family.
    """

    def __init__(self, df, text_col, label_col, id_col, tokenizer, max_len=1024):
        self.df = df.reset_index(drop=True)
        self.text_col = text_col
        self.label_col = label_col
        self.id_col = id_col
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]

        text = r[self.text_col] if pd.notna(r[self.text_col]) else ""
        y = int(r[self.label_col])
        pid = str(r[self.id_col])

        enc = self.tokenizer(
            str(text),
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "paper_id": pid,
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(y, dtype=torch.long),
        }


def collate_longformer(batch):
    return {
        "paper_ids": [b["paper_id"] for b in batch],
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "labels": torch.stack([b["labels"] for b in batch]),
    }


@torch.no_grad()
def predict_longformer_probs(model, loader):
    model.eval()

    all_ids, all_y, all_p = [], [], []

    pbar = tqdm(loader, total=len(loader), desc="Longformer predicting", leave=False)

    for batch in pbar:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].cpu().numpy()

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).logits

        probs = F.softmax(logits, dim=-1)[:, 1].detach().cpu().numpy()

        all_ids.extend(batch["paper_ids"])
        all_y.extend(labels.tolist())
        all_p.extend(probs.tolist())

    return all_ids, np.array(all_y, dtype=int), np.array(all_p, dtype=float)


def train_longformer_one_epoch(model, loader, optim, scheduler=None, grad_accum_steps=8):
    model.train()
    total_loss = 0.0
    optim.zero_grad(set_to_none=True)

    pbar = tqdm(loader, total=len(loader), desc="Longformer training", leave=True)

    for step, batch in enumerate(pbar, start=1):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            out = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = out.loss / grad_accum_steps

        scaler.scale(loss).backward()

        if step % grad_accum_steps == 0:
            scaler.unscale_(optim)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optim)
            scaler.update()
            optim.zero_grad(set_to_none=True)

            if scheduler:
                scheduler.step()

        batch_loss = float(loss.item()) * grad_accum_steps
        total_loss += batch_loss

        if step == 1 or step % 10 == 0:
            pbar.set_postfix({"loss": f"{batch_loss:.4f}", "step": step})

    if (len(loader) % grad_accum_steps) != 0:
        scaler.unscale_(optim)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optim)
        scaler.update()
        optim.zero_grad(set_to_none=True)

        if scheduler:
            scheduler.step()

    return total_loss / max(1, len(loader))


def run_longformer_experiment(
    text_col,
    epochs=1,
    batch_size=1,
    grad_accum_steps=8,
    lr=1e-5,
    max_len=LONG_MAX_LEN
):
    train_df_l = analysis_df[analysis_df["_split"] == "train"].copy()
    val_df_l   = analysis_df[analysis_df["_split"] == "val"].copy()
    test_df_l  = analysis_df[analysis_df["_split"] == "test"].copy()

    print("\n=== Longformer experiment ===")
    print("text_col:", text_col)
    print("train:", len(train_df_l), "val:", len(val_df_l), "test:", len(test_df_l))
    print("max_len:", max_len)
    print("device:", DEVICE)

    long_tokenizer = AutoTokenizer.from_pretrained(LONG_MODEL_NAME)
    long_model = AutoModelForSequenceClassification.from_pretrained(
        LONG_MODEL_NAME,
        num_labels=2
    ).to(DEVICE)

    ds_train = LongTextDataset(
        train_df_l,
        text_col,
        COLS["label"],
        COLS["paper_id"],
        long_tokenizer,
        max_len=max_len
    )

    ds_val = LongTextDataset(
        val_df_l,
        text_col,
        COLS["label"],
        COLS["paper_id"],
        long_tokenizer,
        max_len=max_len
    )

    ds_test = LongTextDataset(
        test_df_l,
        text_col,
        COLS["label"],
        COLS["paper_id"],
        long_tokenizer,
        max_len=max_len
    )

    dl_train = DataLoader(
        ds_train,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_longformer,
        num_workers=0
    )

    dl_val = DataLoader(
        ds_val,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_longformer,
        num_workers=0
    )

    dl_test = DataLoader(
        ds_test,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_longformer,
        num_workers=0
    )

    optim = torch.optim.AdamW(long_model.parameters(), lr=lr)

    num_steps = epochs * max(1, len(dl_train))

    scheduler = get_linear_schedule_with_warmup(
        optim,
        num_warmup_steps=max(10, int(0.05 * num_steps)),
        num_training_steps=num_steps
    )

    history = {
        "model_name": LONG_MODEL_NAME,
        "text_col": text_col,
        "max_len": max_len,
        "epochs": epochs,
        "batch_size": batch_size,
        "grad_accum_steps": grad_accum_steps,
        "train_loss": [],
        "val": [],
        "test": []
    }

    for ep in range(1, epochs + 1):
        print(f"\n--- Longformer epoch {ep}/{epochs} ---")

        loss = train_longformer_one_epoch(
            long_model,
            dl_train,
            optim,
            scheduler=scheduler,
            grad_accum_steps=grad_accum_steps
        )

        history["train_loss"].append(float(loss))

        _, yv, pv = predict_longformer_probs(long_model, dl_val)
        _, yt, pt = predict_longformer_probs(long_model, dl_test)

        val_metrics = eval_probs(yv, pv)
        test_metrics = eval_probs(yt, pt)

        history["val"].append(val_metrics)
        history["test"].append(test_metrics)

        print("Longformer val:", val_metrics)
        print("Longformer test:", test_metrics)

    safe_name = text_col.replace("_doc_", "").replace("_only", "")
    long_dir = os.path.join(RUN_DIR, f"ckpt_longformer_{safe_name}")

    os.makedirs(long_dir, exist_ok=True)

    long_model.save_pretrained(long_dir)
    long_tokenizer.save_pretrained(long_dir)

    save_json(
        history,
        os.path.join(RUN_DIR, f"history_longformer_{text_col.replace('_doc_', '')}.json")
    )

    row = {
        "run_ts": now_str(),
        "exp_name": EXP_NAME,
        "model_name": LONG_MODEL_NAME,
        "ckpt_dir": long_dir,
        "kind": "long_context",
        "name": f"longformer_{text_col.replace('_doc_', '')}",
        "text_col": text_col,
        "agg": f"long_context_{LONG_MAX_LEN}",
        "val_AUPRC": val_metrics["AUPRC"],
        "val_ROC_AUC": val_metrics["ROC_AUC"],
        "val_Brier": val_metrics["Brier"],
        "test_AUPRC": test_metrics["AUPRC"],
        "test_ROC_AUC": test_metrics["ROC_AUC"],
        "test_Brier": test_metrics["Brier"],
        "delta_val_AUPRC_vs_full": np.nan,
        "delta_test_AUPRC_vs_full": np.nan,
        "delta_val_Brier_vs_full": np.nan,
        "delta_test_Brier_vs_full": np.nan,
        "n_val_docs": int(len(val_df_l)),
        "n_test_docs": int(len(test_df_l)),
    }

    append_csv_row(RESULTS_PATH, row, header_order=COL_ORDER)

    del ds_train, ds_val, ds_test, dl_train, dl_val, dl_test
    _cleanup_cuda()

    return long_model, long_tokenizer, history, long_dir


LONGFORMER_CKPTS = {}

if RUN_LONGFORMER:
    for lf_col in LONGFORMER_TEXT_COLS:
        if lf_col not in analysis_df.columns:
            print(f"Skipping Longformer for missing column: {lf_col}")
            continue

        lf_model, lf_tokenizer, lf_history, lf_ckpt = run_longformer_experiment(
            text_col=lf_col,
            epochs=1,
            batch_size=1,
            grad_accum_steps=8,
            lr=1e-5,
            max_len=LONG_MAX_LEN
        )

        LONGFORMER_CKPTS[lf_col] = lf_ckpt

        print("Longformer checkpoint:", lf_ckpt)

        # Do not keep all Longformer models in memory
        del lf_model, lf_tokenizer, lf_history
        _cleanup_cuda()

    save_json(
        LONGFORMER_CKPTS,
        os.path.join(RUN_DIR, "longformer_checkpoints.json")
    )

    print("Saved Longformer checkpoint map:", os.path.join(RUN_DIR, "longformer_checkpoints.json"))

else:
    print("Longformer skipped because RUN_LONGFORMER=False")

PART G — XAI summaries, calibration, ECE, and reliability analysis

In [ ]:
# CELL 31) Faithfulness audit runner for selected experiments

AUDIT_K_FRACS = (0.05, 0.10, 0.20)
AUDIT_N_STEPS = 12
AUDIT_N_DOCS = 30

def run_faithfulness_for_col(
    tag,
    text_col,
    n_docs=AUDIT_N_DOCS,
    k_fracs=AUDIT_K_FRACS,
    n_steps=AUDIT_N_STEPS,
    seed=SEED
):
    df_subset = test_df.copy()

    audit_df = faithfulness_audit(
        df_subset,
        text_col,
        n=min(n_docs, len(df_subset)),
        k_fracs=k_fracs,
        n_steps=n_steps,
        seed=seed
    )

    if len(audit_df) == 0:
        cols = ["paper_id", "text_col", "p0"]
        for kf in k_fracs:
            cols += [f"k_{kf}", f"p_mask_{kf}", f"delta_{kf}"]
        audit_df = pd.DataFrame(columns=cols)

    audit_df["run_ts"] = now_str()
    audit_df["tag"] = tag
    audit_df["model_name"] = MODEL_NAME
    audit_df["ckpt_dir"] = CKPT_DIR

    return audit_df


res = pd.read_csv(RESULTS_PATH)

top_masks = (
    res[(res["kind"] == "mask") & (res["agg"] == "mean")]
    .sort_values("delta_test_AUPRC_vs_full")
    .head(2)
)

audit_targets = [
    ("baseline_full", "_doc_full"),
    ("abstract_only", "_doc_abstract_only"),
    ("discussion_only", "_doc_discussion_only"),
]

for _, r in top_masks.iterrows():
    audit_targets.append((f"mask_{r['name']}", r["text_col"]))

audit_targets = list(dict.fromkeys(audit_targets))

print("Audit targets:")
for t in audit_targets:
    print(" -", t)


base_cols = ["run_ts", "tag", "model_name", "ckpt_dir", "paper_id", "text_col", "p0"]

k_cols = []
for kf in AUDIT_K_FRACS:
    k_cols += [f"k_{kf}", f"p_mask_{kf}", f"delta_{kf}"]

ordered_cols = base_cols + k_cols


# ---------------------------------------------------------
# Resume logic
# ---------------------------------------------------------

if os.path.exists(XAI_AUDIT_PATH):
    try:
        audit_all = pd.read_csv(XAI_AUDIT_PATH)

        for c in ordered_cols:
            if c not in audit_all.columns:
                audit_all[c] = np.nan

        audit_all = audit_all[ordered_cols]

        completed_tags = set(audit_all["tag"].dropna().astype(str).unique())

        print(f"\nLoaded existing XAI audit: {len(audit_all)} rows")
        print("Completed tags:", sorted(completed_tags))

    except Exception as e:
        raise RuntimeError(
            f"Could not read existing XAI_AUDIT_PATH:\n{XAI_AUDIT_PATH}\n"
            f"Error: {e}\n\n"
            "If this file is corrupted, delete ONLY this file and rerun Cell 31."
        )
else:
    audit_all = pd.DataFrame(columns=ordered_cols)
    completed_tags = set()
    print("\nNo existing XAI audit found. Starting fresh.")


# ---------------------------------------------------------
# Run only missing targets; save after each target
# ---------------------------------------------------------

for tag, col in audit_targets:
    if tag in completed_tags:
        print(f"Skipping already completed faithfulness audit: {tag}")
        continue

    if col not in test_df.columns:
        print(f"Skipping {tag}: column not found -> {col}")
        continue

    print(f"\nRunning faithfulness audit: {tag} ({col})")

    a = run_faithfulness_for_col(tag, col)

    for c in ordered_cols:
        if c not in a.columns:
            a[c] = np.nan

    a = a[ordered_cols]

    audit_all = pd.concat([audit_all, a], ignore_index=True)

    # Save immediately after each completed target
    audit_all.to_csv(XAI_AUDIT_PATH, index=False)

    completed_tags.add(tag)

    print(f"Saved partial XAI audit after {tag}: {XAI_AUDIT_PATH}")


if len(audit_all) == 0:
    raise RuntimeError("No audits were produced.")


# Final clean save
audit_all = audit_all[ordered_cols]
audit_all.to_csv(XAI_AUDIT_PATH, index=False)

print("\nSaved clean XAI audit to:", XAI_AUDIT_PATH)

for kf in AUDIT_K_FRACS:
    col = f"delta_{kf}"
    print(
        f"Mean ΔP @ {kf}: {audit_all[col].mean():.4f} | "
        f"Median: {audit_all[col].median():.4f} | "
        f"n={audit_all[col].notna().sum()}"
    )

display(audit_all.head())

In [ ]:
# CELL 32) Summarise faithfulness results (per experiment tag) (UPDATED for multi-k_fracs output)

xai = pd.read_csv(XAI_AUDIT_PATH)

# Identify k_fracs from columns dynamically (robust)
delta_cols = [c for c in xai.columns if c.startswith("delta_")]
pmask_cols = [c for c in xai.columns if c.startswith("p_mask_")]

if len(delta_cols) == 0:
    raise RuntimeError("No delta_* columns found in XAI_AUDIT_PATH. Did Cell 31 run with the updated schema?")

def _mean(series):
    return float(np.nanmean(series)) if series.notna().any() else np.nan

def _median(series):
    return float(np.nanmedian(series)) if series.notna().any() else np.nan

group_cols = ["tag", "text_col"]

# Base summary
summary = (
    xai.groupby(group_cols)
       .agg(
           n=("paper_id", "count"),
           mean_p0=("p0", _mean),
           median_p0=("p0", _median),
       )
       .reset_index()
)

# Add per-k_frac metrics (mean/median for each delta and p_mask)
for c in delta_cols:
    summary[f"mean_{c}"] = xai.groupby(group_cols)[c].apply(_mean).values
    summary[f"median_{c}"] = xai.groupby(group_cols)[c].apply(_median).values

for c in pmask_cols:
    summary[f"mean_{c}"] = xai.groupby(group_cols)[c].apply(_mean).values
    summary[f"median_{c}"] = xai.groupby(group_cols)[c].apply(_median).values

# Sort by the "strongest degradation" if possible (most negative mean delta at smallest k)
# Heuristic: pick the smallest k_frac present
def _parse_k(colname):
    # colname like "delta_0.05"
    try:
        return float(colname.split("_", 1)[1])
    except Exception:
        return None

parsed = [(c, _parse_k(c)) for c in delta_cols]
parsed = [(c, k) for c, k in parsed if k is not None]
if len(parsed):
    smallest = sorted(parsed, key=lambda x: x[1])[0][0]  # e.g., delta_0.05
    sort_col = f"mean_{smallest}"
    summary = summary.sort_values(sort_col, ascending=True)

summary_path = os.path.join(EXP_DIR, "xai_faithfulness_summary.csv")
summary.to_csv(summary_path, index=False)

print("Saved:", summary_path)
display(summary)

In [ ]:
# CELL 33) Calibration helpers: Platt scaling, isotonic regression, and reliability curves

def fit_platt_scaler(p_val, y_val):
    """
    Platt scaling: fit logistic regression on validation probabilities.
    Input p_val: shape (n,)
    """
    lr = LogisticRegression(solver="lbfgs")
    lr.fit(p_val.reshape(-1, 1), y_val.astype(int))
    return lr

def apply_platt_scaler(platt_model, p):
    return platt_model.predict_proba(p.reshape(-1, 1))[:, 1]

def fit_isotonic_scaler(p_val, y_val):
    """
    Isotonic regression calibration.
    """
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(p_val, y_val.astype(int))
    return iso

def apply_isotonic_scaler(iso_model, p):
    return iso_model.predict(p)

def plot_reliability(y_true, p, title, n_bins=10):
    frac_pos, mean_pred = calibration_curve(y_true, p, n_bins=n_bins, strategy="quantile")
    plt.figure()
    plt.plot(mean_pred, frac_pos, marker="o")
    plt.plot([0,1],[0,1], linestyle="--")
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Fraction of positives")
    plt.title(title)
    plt.grid(True)
    plt.show()

In [ ]:
# CELL 34) Get probabilities and metrics for any text column

CACHE_MASKED = False  # set True only if you really want masked views cached on Drive

@torch.no_grad()
def get_doc_probs_for_col(df_subset, text_col, agg="mean", batch_docs=2):
    view_name = text_col.replace("_doc_", "")

    allow_cache = True
    if (not CACHE_MASKED) and text_col.startswith("_doc_full__"):
        allow_cache = False

    max_chunks = VIEW_MAX_CHUNKS.get(text_col, MAX_CHUNKS_PER_DOC)

    ds = CachedChunkedDocDataset(
        df_subset,
        text_col,
        COLS["label"],
        COLS["paper_id"],
        view_name=view_name,
        max_len=MAX_LEN,
        stride=STRIDE,
        max_chunks=max_chunks,
        allow_build_cache=allow_cache
    )

    dl = DataLoader(
        ds,
        batch_size=batch_docs,
        shuffle=False,
        collate_fn=collate_cached_chunked
    )

    doc_ids, y_true, y_prob = predict_doc_probs(model, dl, agg=agg)
    return doc_ids, y_true, y_prob


@torch.no_grad()
def eval_transformer_on_col(df_subset, text_col, agg="mean", batch_docs=2):
    _, y_true, y_prob = get_doc_probs_for_col(
        df_subset,
        text_col,
        agg=agg,
        batch_docs=batch_docs
    )

    return eval_probs(y_true, y_prob, name=text_col), y_true, y_prob


train_df = analysis_df[analysis_df["_split"] == "train"].copy()
val_df   = analysis_df[analysis_df["_split"] == "val"].copy()
test_df  = analysis_df[analysis_df["_split"] == "test"].copy()

base_metrics_test, base_y, base_p = eval_transformer_on_col(
    test_df,
    "_doc_full",
    agg="mean",
    batch_docs=2
)

print("BASE test metrics:", base_metrics_test)
save_json({"base_test": base_metrics_test}, os.path.join(RUN_DIR, "masking_base_metrics.json"))

In [ ]:
# CELL 35) Calibration run for one experiment (full doc, mean agg) + save artifacts

CAL_TEXT_COL = "_doc_full"
CAL_AGG = "mean"

val_doc_ids, yv, pv = get_doc_probs_for_col(val_df, CAL_TEXT_COL, agg=CAL_AGG, batch_docs=2)
test_doc_ids, yt, pt = get_doc_probs_for_col(test_df, CAL_TEXT_COL, agg=CAL_AGG, batch_docs=2)

# Uncalibrated metrics
m_test_uncal = eval_probs(yt, pt, name="test_uncal")
print("TEST uncal:", m_test_uncal)

# Fit calibrators on VAL
platt = fit_platt_scaler(pv, yv)
iso = fit_isotonic_scaler(pv, yv)

pt_platt = apply_platt_scaler(platt, pt)
pt_iso   = apply_isotonic_scaler(iso, pt)

m_test_platt = eval_probs(yt, pt_platt, name="test_platt")
m_test_iso   = eval_probs(yt, pt_iso, name="test_iso")

print("TEST platt:", m_test_platt)
print("TEST iso  :", m_test_iso)

# Save calibration artifacts
cal_dir = os.path.join(EXP_DIR, "calibration_full_mean")
os.makedirs(cal_dir, exist_ok=True)

save_json({
    "text_col": CAL_TEXT_COL,
    "agg": CAL_AGG,
    "test_uncal": m_test_uncal,
    "test_platt": m_test_platt,
    "test_iso": m_test_iso
}, os.path.join(cal_dir, "calibration_metrics.json"))

# Reliability plots (optional but very useful)
plot_reliability(yt, pt, "Reliability (Uncalibrated) - FULL/mean")
plot_reliability(yt, pt_platt, "Reliability (Platt) - FULL/mean")
plot_reliability(yt, pt_iso, "Reliability (Isotonic) - FULL/mean")

In [ ]:
# CELL 36) Add calibrated results back into results_master.csv

def write_calibrated_rows(kind, name, text_col, agg, base_row_meta, metrics_uncal, metrics_platt, metrics_iso):
    base = baseline_by_agg[agg]  # full baseline for deltas

    def row_for(cal_name, m):
        return {
            "run_ts": now_str(),
            **base_row_meta,
            "kind": kind,
            "name": f"{name}__{cal_name}",
            "text_col": text_col,
            "agg": agg,
            "val_AUPRC": np.nan,
            "val_ROC_AUC": np.nan,
            "val_Brier": np.nan,
            "test_AUPRC": m["AUPRC"],
            "test_ROC_AUC": m["ROC_AUC"],
            "test_Brier": m["Brier"],
            "delta_val_AUPRC_vs_full": np.nan,
            "delta_test_AUPRC_vs_full": m["AUPRC"] - base["test_AUPRC"],   # compare vs full baseline test
            "delta_val_Brier_vs_full": np.nan,
            "delta_test_Brier_vs_full": m["Brier"] - base["test_Brier"],
            "n_val_docs": int(len(np.unique(val_df[COLS["paper_id"]]))),
            "n_test_docs": int(len(np.unique(test_df[COLS["paper_id"]]))),
        }

    append_csv_row(RESULTS_PATH, row_for("uncal", metrics_uncal), header_order=COL_ORDER)
    append_csv_row(RESULTS_PATH, row_for("platt", metrics_platt), header_order=COL_ORDER)
    append_csv_row(RESULTS_PATH, row_for("isotonic", metrics_iso), header_order=COL_ORDER)

BASE_META = {"exp_name": EXP_NAME, "model_name": MODEL_NAME, "ckpt_dir": CKPT_DIR}

write_calibrated_rows(
    kind="view",
    name="full",
    text_col=CAL_TEXT_COL,
    agg=CAL_AGG,
    base_row_meta=BASE_META,
    metrics_uncal=m_test_uncal,
    metrics_platt=m_test_platt,
    metrics_iso=m_test_iso
)

print("Calibration rows appended to:", RESULTS_PATH)

PART H — Temporal split, classical model extensions, and dataset freeze

In [ ]:
# CELL 37) Temporal split definition using article year

YEAR_COL = "year"

if YEAR_COL not in analysis_df.columns:
    print(f"Year column '{YEAR_COL}' not found. Skipping temporal split.")
    analysis_df["_split_time"] = "unknown_year"
else:
    # Coerce year to numeric safely
    year_num = pd.to_numeric(analysis_df[YEAR_COL], errors="coerce")

    # Only valid years used to compute cutoff
    valid = year_num.dropna()
    if len(valid) < 50:
        print(f"Too few valid years ({len(valid)}) to do a meaningful temporal split. Marking all as unknown_year.")
        analysis_df["_split_time"] = "unknown_year"
    else:
        # Use a percentile cutoff (train on earlier years)
        cut_q = 0.70
        cutoff_year = int(np.quantile(valid.values.astype(float), cut_q))

        analysis_df["_split_time"] = "unknown_year"
        analysis_df.loc[year_num.notna(), "_split_time"] = np.where(
            year_num[year_num.notna()].astype(int) <= cutoff_year,
            "train_time",
            "test_time"
        )

        print("Temporal cutoff_year:", cutoff_year)
        print(analysis_df["_split_time"].value_counts(dropna=False))

In [ ]:
# CELL 38) Classical baselines: TF-IDF word n-grams + character n-grams

def eval_tfidf_lr(
    text_col,
    analyzer="word",
    ngram=(1, 2),
    min_df=3,
    max_df=0.95,
    max_features=None
):
    train_c = analysis_df[analysis_df["_split"] == "train"].copy()
    val_c   = analysis_df[analysis_df["_split"] == "val"].copy()
    test_c  = analysis_df[analysis_df["_split"] == "test"].copy()

    X_train = train_c[text_col].fillna("")
    y_train = train_c[COLS["label"]].astype(int)

    X_val = val_c[text_col].fillna("")
    y_val = val_c[COLS["label"]].astype(int)

    X_test = test_c[text_col].fillna("")
    y_test = test_c[COLS["label"]].astype(int)

    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(
            analyzer=analyzer,
            ngram_range=ngram,
            min_df=min_df,
            max_df=max_df,
            max_features=max_features,
            strip_accents="unicode",
            lowercase=True
        )),
        ("clf", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            n_jobs=-1
        ))
    ])

    pipe.fit(X_train, y_train)

    pv = pipe.predict_proba(X_val)[:, 1]
    pt = pipe.predict_proba(X_test)[:, 1]

    m_val = eval_probs(y_val, pv, "val")
    m_test = eval_probs(y_test, pt, "test")

    return pipe, m_val, m_test


def write_classical_row(kind, name, text_col, m_val, m_test, feature_type):
    row = {
        "run_ts": now_str(),
        "exp_name": EXP_NAME,
        "model_name": "TFIDF+LogReg",
        "ckpt_dir": "",
        "kind": kind,
        "name": name,
        "text_col": text_col,
        "agg": feature_type,
        "val_AUPRC": m_val["AUPRC"],
        "val_ROC_AUC": m_val["ROC_AUC"],
        "val_Brier": m_val["Brier"],
        "test_AUPRC": m_test["AUPRC"],
        "test_ROC_AUC": m_test["ROC_AUC"],
        "test_Brier": m_test["Brier"],
        "delta_val_AUPRC_vs_full": np.nan,
        "delta_test_AUPRC_vs_full": np.nan,
        "delta_val_Brier_vs_full": np.nan,
        "delta_test_Brier_vs_full": np.nan,
        "n_val_docs": int(len(np.unique(val_df[COLS["paper_id"]]))),
        "n_test_docs": int(len(np.unique(test_df[COLS["paper_id"]]))),
    }

    append_csv_row(RESULTS_PATH, row, header_order=COL_ORDER)


classical_models_dir = os.path.join(EXP_DIR, "classical_models")
os.makedirs(classical_models_dir, exist_ok=True)

CLASSICAL_TEXT_COLS = [
    c for c in [
        "_doc_full",
        "_doc_core",
        "_doc_title_only",
        "_doc_abstract_only",
        "_doc_introduction_only",
        "_doc_methods_only",
        "_doc_results_only",
        "_doc_discussion_only",
    ]
    if c in analysis_df.columns
]

classical_rows = []

for text_col in CLASSICAL_TEXT_COLS:

    pipe_word, m_val_word, m_test_word = eval_tfidf_lr(
        text_col=text_col,
        analyzer="word",
        ngram=(1, 2),
        min_df=3,
        max_df=0.95
    )

    name_word = f"tfidf_word_{text_col.replace('_doc_', '')}"

    write_classical_row(
        kind="classical",
        name=name_word,
        text_col=text_col,
        m_val=m_val_word,
        m_test=m_test_word,
        feature_type="word_1_2"
    )

    joblib.dump(pipe_word, os.path.join(classical_models_dir, f"{name_word}.joblib"))

    classical_rows.append({
        "feature_type": "word_1_2",
        "text_col": text_col,
        "val_AUPRC": m_val_word["AUPRC"],
        "test_AUPRC": m_test_word["AUPRC"],
        "test_ROC_AUC": m_test_word["ROC_AUC"],
        "test_Brier": m_test_word["Brier"],
    })

    pipe_char, m_val_char, m_test_char = eval_tfidf_lr(
        text_col=text_col,
        analyzer="char_wb",
        ngram=(3, 5),
        min_df=3,
        max_df=0.95,
        max_features=300000
    )

    name_char = f"tfidf_char_{text_col.replace('_doc_', '')}"

    write_classical_row(
        kind="classical",
        name=name_char,
        text_col=text_col,
        m_val=m_val_char,
        m_test=m_test_char,
        feature_type="char_wb_3_5"
    )

    joblib.dump(pipe_char, os.path.join(classical_models_dir, f"{name_char}.joblib"))

    classical_rows.append({
        "feature_type": "char_wb_3_5",
        "text_col": text_col,
        "val_AUPRC": m_val_char["AUPRC"],
        "test_AUPRC": m_test_char["AUPRC"],
        "test_ROC_AUC": m_test_char["ROC_AUC"],
        "test_Brier": m_test_char["Brier"],
    })

classical_summary = pd.DataFrame(classical_rows)

classical_summary_path = os.path.join(
    EXP_DIR,
    "classical_tfidf_word_char_summary.csv"
)

classical_summary.to_csv(classical_summary_path, index=False)

print("Saved classical TF-IDF word/char summary:", classical_summary_path)
display(classical_summary.sort_values("test_AUPRC", ascending=False))

In [ ]:
# CELL 39) Inspect consolidated leaderboard: SciBERT, calibrated rows, and classical baselines

res_all = pd.read_csv(RESULTS_PATH)

print("=== Top 15 by test AUPRC (all models) ===")
display(res_all.sort_values("test_AUPRC", ascending=False).head(15))

print("\n=== Filter: SciBERT only ===")
display(res_all[res_all["model_name"]==MODEL_NAME].sort_values("test_AUPRC", ascending=False).head(15))

print("\n=== Filter: Classical TF-IDF+LR only ===")
display(res_all[res_all["model_name"]=="TFIDF+LogReg"].sort_values("test_AUPRC", ascending=False).head(15))

In [ ]:
# CELL 40) FINAL DATASET FREEZE & METADATA SNAPSHOT

FINAL_DB_PATH = os.path.join(RUN_DIR, "preprocessed_db_final_for_modelling_FREEZE.csv")
analysis_df.to_csv(FINAL_DB_PATH, index=False)
print("Final dataset frozen at:", FINAL_DB_PATH)

def safe_year_range(df, col="year"):
    if col not in df.columns:
        return None, None
    y = pd.to_numeric(df[col], errors="coerce")
    y = y.dropna()
    if len(y) == 0:
        return None, None
    return int(y.min()), int(y.max())

year_min, year_max = safe_year_range(analysis_df, col="year")

meta = {
    "run_id": RUN_ID,
    "timestamp_utc": now_str(),
    "n_rows_total": int(len(analysis_df)),
    "n_cases": int((analysis_df[COLS["label"]] == 1).sum()),
    "n_controls": int((analysis_df[COLS["label"]] == 0).sum()),
    "year_min": year_min,
    "year_max": year_max,
    "missing_rate_by_section": {
        s: float((analysis_df.get(f"_clean_{s}", pd.Series([""]*len(analysis_df))).fillna("").str.len() == 0).mean())
        for s in SECTIONS
    },
    "splits": analysis_df["_split"].value_counts(dropna=False).to_dict()
}

save_json(meta, os.path.join(RUN_DIR, "dataset_metadata.json"))
print("Dataset metadata saved.")

PART I — Final dissertation tables, artefact freezing, and XAI reporting

In [ ]:
# CELL 41) SELECT FINAL RESULTS

res_all = pd.read_csv(RESULTS_PATH)

FINAL_KEEP = (
    # SciBERT main mean-aggregation rows
    (
        (res_all["model_name"] == MODEL_NAME) &
        (res_all["agg"] == "mean") &
        (res_all["kind"].isin(["view", "mask"]))
    )
    |
    # TF-IDF word/char rows
    (
        (res_all["model_name"] == "TFIDF+LogReg") &
        (res_all["kind"] == "classical")
    )
    |
    # Longformer rows
    (
        (res_all["model_name"] == LONG_MODEL_NAME) &
        (res_all["kind"] == "long_context")
    )
)

final_res = res_all[FINAL_KEEP].copy()

final_res = final_res.sort_values(
    ["model_name", "kind", "test_AUPRC"],
    ascending=[True, True, False]
)

FINAL_RESULTS_PATH = os.path.join(RUN_DIR, "final_results_table.csv")
final_res.to_csv(FINAL_RESULTS_PATH, index=False)

print("Final results table saved:")
print(FINAL_RESULTS_PATH)

display(final_res)

In [ ]:
# CELL 42) SECTION CONTRIBUTION TABLE

mask_res = res_all[
    (res_all["kind"] == "mask") &
    (res_all["agg"] == "mean")
].copy()

mask_res = mask_res.sort_values("delta_test_AUPRC_vs_full")

MASK_TABLE_PATH = os.path.join(RUN_DIR, "final_section_ablation_table.csv")
mask_res.to_csv(MASK_TABLE_PATH, index=False)

print("Section ablation table saved:")
print(MASK_TABLE_PATH)

display(mask_res[[
    "name",
    "test_AUPRC",
    "delta_test_AUPRC_vs_full",
    "test_Brier"
]])

In [ ]:
# CELL 43) EXPLAINABILITY EXPORT

# Where the XAI files actually live (based on earlier cells)
# Prefer the per-experiment xai folder
xai_dir = os.path.join(EXP_DIR, "xai")

# 1) Freeze single example
XAI_SINGLE_FINAL = os.path.join(RUN_DIR, "xai_example_FINAL.json")

# Candidate locations (try in order)
candidates = [
    os.path.join(xai_dir, "xai_faithfulness_single_example.json"),
    os.path.join(EXP_DIR, "xai_faithfulness_single_example.json"),
    os.path.join(RUN_DIR, "xai_faithfulness_single_example.json"),
]

# As a fallback: find *any* matching file under EXP_DIR
if not any(os.path.exists(p) for p in candidates):
    hits = glob.glob(os.path.join(EXP_DIR, "**", "xai_faithfulness_single_example.json"), recursive=True)
    if hits:
        candidates.insert(0, hits[0])

src = next((p for p in candidates if os.path.exists(p)), None)
if src is None:
    raise FileNotFoundError(
        "Could not locate xai_faithfulness_single_example.json.\n"
        f"Tried:\n- " + "\n- ".join(candidates)
    )

shutil.copy(src, XAI_SINGLE_FINAL)
print(" Local XAI example frozen:")
print("   src:", src)
print("   dst:", XAI_SINGLE_FINAL)

# 2) Freeze XAI summary
summary_candidates = [
    os.path.join(EXP_DIR, "xai_faithfulness_summary.csv"),
    os.path.join(xai_dir, "xai_faithfulness_summary.csv"),
]

summary_src = next((p for p in summary_candidates if os.path.exists(p)), None)
if summary_src is None:
    hits = glob.glob(os.path.join(EXP_DIR, "**", "xai_faithfulness_summary.csv"), recursive=True)
    if hits:
        summary_src = hits[0]

if summary_src is None:
    raise FileNotFoundError(
        "Could not locate xai_faithfulness_summary.csv under EXP_DIR.\n"
        f"Tried:\n- " + "\n- ".join(summary_candidates)
    )

xai_summary = pd.read_csv(summary_src)

XAI_SUMMARY_FINAL = os.path.join(RUN_DIR, "xai_faithfulness_summary_FINAL.csv")
xai_summary.to_csv(XAI_SUMMARY_FINAL, index=False)

print(" XAI faithfulness summary saved:")
print("   src:", summary_src)
print("   dst:", XAI_SUMMARY_FINAL)

display(xai_summary)

In [ ]:
# CELL 44) CALIBRATION FIGURES & METRICS

CAL_FINAL_DIR = os.path.join(RUN_DIR, "calibration_FINAL")
os.makedirs(CAL_FINAL_DIR, exist_ok=True)

shutil.copy(
    os.path.join(EXP_DIR, "calibration_full_mean", "calibration_metrics.json"),
    os.path.join(CAL_FINAL_DIR, "calibration_metrics_FINAL.json")
)

print("Calibration metrics frozen.")

In [ ]:
# CELL 45) Export Integrated Gradients local explanations as table-ready CSV

def ig_top_tokens_for_text(text, paper_id, text_col, label=None, top_n=30, n_steps=24):
    tokens, scores, input_ids, attention_mask = get_token_attributions(text, max_len=MAX_LEN, n_steps=n_steps)
    p0 = prob_case_from_encoding(input_ids, attention_mask)
    rows = []
    special = set(tokenizer.all_special_tokens)
    for pos, (tok, score) in enumerate(zip(tokens, scores)):
        if tok in special or tok == tokenizer.pad_token:
            continue
        rows.append({
            "paper_id": str(paper_id),
            "text_col": text_col,
            "true_label": label,
            "p_retracted": float(p0),
            "token_position": int(pos),
            "token": tok,
            "attribution": float(score),
            "abs_attribution": float(abs(score)),
        })
    out = pd.DataFrame(rows).sort_values("abs_attribution", ascending=False).head(top_n)
    return out

IG_EXPORT_DIR = os.path.join(EXP_DIR, "xai_exports")
os.makedirs(IG_EXPORT_DIR, exist_ok=True)

XAI_TEXT_COLS = [c for c in ["_doc_abstract_only", "_doc_discussion_only", "_doc_methods_only", "_doc_full"] if c in analysis_df.columns]
XAI_SAMPLE_N = 10
xai_source = analysis_df[analysis_df["_split"] == "test"].copy()
xai_source = xai_source[xai_source[COLS["label"]] == 1].copy() if (xai_source[COLS["label"]] == 1).any() else xai_source
xai_sample = xai_source.sample(min(XAI_SAMPLE_N, len(xai_source)), random_state=SEED)

ig_rows = []
for _, r in xai_sample.iterrows():
    for col in XAI_TEXT_COLS:
        txt = r[col] if pd.notna(r[col]) else ""
        if str(txt).strip():
            ig_rows.append(ig_top_tokens_for_text(txt, r[COLS["paper_id"]], col, int(r[COLS["label"]]), top_n=30, n_steps=12))

ig_export = pd.concat(ig_rows, ignore_index=True) if ig_rows else pd.DataFrame()
ig_path = os.path.join(IG_EXPORT_DIR, "integrated_gradients_top_tokens.csv")
ig_export.to_csv(ig_path, index=False)
print("Saved:", ig_path)
display(ig_export.head(30))

In [ ]:
# CELL 46) Section-level faithfulness summary from XAI audit

if not os.path.exists(XAI_AUDIT_PATH):
    raise FileNotFoundError(
        f"XAI audit file not found: {XAI_AUDIT_PATH}. Run Cell 24 first."
    )

xai_audit = pd.read_csv(XAI_AUDIT_PATH)

# Detect delta columns safely
delta_cols = [c for c in xai_audit.columns if c.startswith("delta_")]

if len(delta_cols) == 0:
    raise RuntimeError(
        "No delta_* columns found in XAI audit file. "
        "Rerun Cell 24 with the corrected version."
    )

# Use text_col as the section/view identifier
if "text_col" not in xai_audit.columns:
    raise RuntimeError("Expected column 'text_col' not found in XAI audit file.")

summary_rows = []

for text_col, part in xai_audit.groupby("text_col"):
    row = {
        "text_col": text_col,
        "n_docs": int(len(part)),
        "mean_p0": float(part["p0"].mean()) if "p0" in part.columns else np.nan,
        "median_p0": float(part["p0"].median()) if "p0" in part.columns else np.nan,
    }

    for c in delta_cols:
        row[f"mean_{c}"] = float(part[c].mean())
        row[f"median_{c}"] = float(part[c].median())
        row[f"std_{c}"] = float(part[c].std())
        row[f"share_negative_{c}"] = float((part[c] < 0).mean())

    summary_rows.append(row)

section_faithfulness = pd.DataFrame(summary_rows)

# Sort by strongest mean probability drop at smallest perturbation fraction
def parse_k(col):
    try:
        return float(col.replace("delta_", ""))
    except Exception:
        return 999

first_delta = sorted(delta_cols, key=parse_k)[0]
sort_col = f"mean_{first_delta}"

if sort_col not in section_faithfulness.columns:
    sort_col = "n_docs"

section_faithfulness = section_faithfulness.sort_values(
    sort_col,
    ascending=True
)

IG_EXPORT_DIR = os.path.join(EXP_DIR, "xai_exports")
os.makedirs(IG_EXPORT_DIR, exist_ok=True)

section_faith_path = os.path.join(
    IG_EXPORT_DIR,
    "section_level_faithfulness_summary.csv"
)

section_faithfulness.to_csv(section_faith_path, index=False)

print("Saved section-level faithfulness summary:", section_faith_path)
display(section_faithfulness)

PART J — LIME, SHAP, extended temporal robustness, and explanation stability

In [ ]:
# CELL 47) LIME local explanations for TF-IDF, SciBERT, and Longformer

IG_EXPORT_DIR = os.path.join(EXP_DIR, "xai_exports")
os.makedirs(IG_EXPORT_DIR, exist_ok=True)

explainer = LimeTextExplainer(class_names=["control", "retracted"])

test_lime = analysis_df[analysis_df["_split"] == "test"].copy()
train_lime = analysis_df[analysis_df["_split"] == "train"].copy()


# -----------------------------
# A. LIME for TF-IDF
# -----------------------------

LIME_VIEW = "_doc_full"

lime_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.95,
        strip_accents="unicode",
        lowercase=True
    )),
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        n_jobs=-1
    ))
])

lime_pipe.fit(
    train_lime[LIME_VIEW].fillna(""),
    train_lime[COLS["label"]].astype(int)
)

lime_rows = []

lime_sample = test_lime.sample(min(10, len(test_lime)), random_state=SEED)

for _, r in lime_sample.iterrows():
    txt = r[LIME_VIEW] if pd.notna(r[LIME_VIEW]) else ""

    if not str(txt).strip():
        continue

    exp = explainer.explain_instance(
        txt,
        lime_pipe.predict_proba,
        num_features=20,
        labels=[1]
    )

    p = float(lime_pipe.predict_proba([txt])[0, 1])

    for term, weight in exp.as_list(label=1):
        lime_rows.append({
            "model_family": "TFIDF+LogReg",
            "paper_id": str(r[COLS["paper_id"]]),
            "text_col": LIME_VIEW,
            "p_retracted": p,
            "term": term,
            "lime_weight": float(weight)
        })

lime_tfidf_export = pd.DataFrame(lime_rows)
lime_tfidf_path = os.path.join(IG_EXPORT_DIR, "lime_local_explanations_tfidf.csv")
lime_tfidf_export.to_csv(lime_tfidf_path, index=False)

print("Saved TF-IDF LIME:", lime_tfidf_path)


# -----------------------------
# B. LIME for SciBERT
# -----------------------------

def scibert_predict_proba_for_lime(texts):
    probs_all = []

    model.eval()

    for text in texts:
        enc = tokenizer(
            text if text else "",
            truncation=True,
            max_length=MAX_LEN,
            padding="max_length",
            return_tensors="pt"
        )

        input_ids = enc["input_ids"].to(DEVICE)
        attention_mask = enc["attention_mask"].to(DEVICE)

        with torch.no_grad():
            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            ).logits

            probs = F.softmax(logits, dim=-1).detach().cpu().numpy()[0]

        probs_all.append(probs)

    return np.asarray(probs_all)


SCIBERT_LIME_VIEW = "_doc_abstract_only"
scibert_lime_rows = []

scibert_source = test_lime.copy()

if (scibert_source[COLS["label"]] == 1).any():
    scibert_source = scibert_source[scibert_source[COLS["label"]] == 1].copy()

scibert_sample = scibert_source.sample(min(5, len(scibert_source)), random_state=SEED)

for _, r in scibert_sample.iterrows():
    txt = r[SCIBERT_LIME_VIEW] if pd.notna(r[SCIBERT_LIME_VIEW]) else ""

    if not str(txt).strip():
        continue

    exp = explainer.explain_instance(
        txt,
        scibert_predict_proba_for_lime,
        num_features=20,
        labels=[1],
        num_samples=500
    )

    p = float(scibert_predict_proba_for_lime([txt])[0, 1])

    for term, weight in exp.as_list(label=1):
        scibert_lime_rows.append({
            "model_family": "SciBERT",
            "paper_id": str(r[COLS["paper_id"]]),
            "text_col": SCIBERT_LIME_VIEW,
            "p_retracted": p,
            "term": term,
            "lime_weight": float(weight)
        })

lime_scibert_export = pd.DataFrame(scibert_lime_rows)
lime_scibert_path = os.path.join(IG_EXPORT_DIR, "lime_local_explanations_scibert.csv")
lime_scibert_export.to_csv(lime_scibert_path, index=False)

print("Saved SciBERT LIME:", lime_scibert_path)


# -----------------------------
# C. LIME for Longformer
# -----------------------------

def longformer_predict_proba_for_lime_factory(lf_model, lf_tokenizer):
    def _predict(texts):
        probs_all = []

        lf_model.eval()

        for text in texts:
            enc = lf_tokenizer(
                text if text else "",
                truncation=True,
                max_length=LONG_MAX_LEN,
                padding="max_length",
                return_tensors="pt"
            )

            input_ids = enc["input_ids"].to(DEVICE)
            attention_mask = enc["attention_mask"].to(DEVICE)

            with torch.no_grad():
                logits = lf_model(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                ).logits

                probs = F.softmax(logits, dim=-1).detach().cpu().numpy()[0]

            probs_all.append(probs)

        return np.asarray(probs_all)

    return _predict


longformer_lime_rows = []

if "LONGFORMER_CKPTS" in globals() and len(LONGFORMER_CKPTS) > 0:

    for lf_view, lf_ckpt in LONGFORMER_CKPTS.items():
        print("Running Longformer LIME for:", lf_view)

        lf_tokenizer = AutoTokenizer.from_pretrained(lf_ckpt)
        lf_model = AutoModelForSequenceClassification.from_pretrained(lf_ckpt).to(DEVICE)

        pred_fn = longformer_predict_proba_for_lime_factory(lf_model, lf_tokenizer)

        lf_source = test_lime.copy()

        if (lf_source[COLS["label"]] == 1).any():
            lf_source = lf_source[lf_source[COLS["label"]] == 1].copy()

        lf_sample = lf_source.sample(min(3, len(lf_source)), random_state=SEED)

        for _, r in lf_sample.iterrows():
            txt = r[lf_view] if pd.notna(r[lf_view]) else ""

            if not str(txt).strip():
                continue

            exp = explainer.explain_instance(
                txt,
                pred_fn,
                num_features=20,
                labels=[1],
                num_samples=300
            )

            p = float(pred_fn([txt])[0, 1])

            for term, weight in exp.as_list(label=1):
                longformer_lime_rows.append({
                    "model_family": "Longformer",
                    "paper_id": str(r[COLS["paper_id"]]),
                    "text_col": lf_view,
                    "p_retracted": p,
                    "term": term,
                    "lime_weight": float(weight)
                })

        del lf_model, lf_tokenizer
        _cleanup_cuda()

lime_longformer_export = pd.DataFrame(longformer_lime_rows)
lime_longformer_path = os.path.join(IG_EXPORT_DIR, "lime_local_explanations_longformer.csv")
lime_longformer_export.to_csv(lime_longformer_path, index=False)

print("Saved Longformer LIME:", lime_longformer_path)

display(pd.concat(
    [
        lime_tfidf_export.head(10),
        lime_scibert_export.head(10),
        lime_longformer_export.head(10)
    ],
    ignore_index=True
))

In [ ]:
# CELL 48) SHAP explanations for TF-IDF, SciBERT, and Longformer

IG_EXPORT_DIR = os.path.join(EXP_DIR, "xai_exports")
os.makedirs(IG_EXPORT_DIR, exist_ok=True)

# -----------------------------
# A. SHAP for TF-IDF
# -----------------------------

SHAP_VIEW = "_doc_full"

train_shap = analysis_df[analysis_df["_split"] == "train"].copy()
test_shap = analysis_df[analysis_df["_split"] == "test"].copy()

shap_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.95,
        strip_accents="unicode",
        lowercase=True
    )),
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        n_jobs=-1
    ))
])

shap_pipe.fit(
    train_shap[SHAP_VIEW].fillna(""),
    train_shap[COLS["label"]].astype(int)
)

vectorizer = shap_pipe.named_steps["tfidf"]
clf = shap_pipe.named_steps["clf"]

X_bg = vectorizer.transform(
    train_shap[SHAP_VIEW].fillna("").sample(
        min(300, len(train_shap)),
        random_state=SEED
    )
)

X_te = vectorizer.transform(
    test_shap[SHAP_VIEW].fillna("").sample(
        min(100, len(test_shap)),
        random_state=SEED
    )
)

expl = shap.LinearExplainer(
    clf,
    X_bg,
    feature_perturbation="interventional"
)

sv = expl.shap_values(X_te)

if isinstance(sv, list):
    sv = sv[1]

mean_abs = np.asarray(np.abs(sv).mean(axis=0)).ravel()
features = np.array(vectorizer.get_feature_names_out())

shap_global = (
    pd.DataFrame({
        "model_family": "TFIDF+LogReg",
        "term": features,
        "mean_abs_shap": mean_abs
    })
    .sort_values("mean_abs_shap", ascending=False)
    .head(100)
)

shap_tfidf_path = os.path.join(
    IG_EXPORT_DIR,
    "shap_global_tfidf_top_terms.csv"
)

shap_global.to_csv(shap_tfidf_path, index=False)

print("Saved TF-IDF SHAP:", shap_tfidf_path)


# -----------------------------
# B. SHAP for SciBERT
# -----------------------------

SCIBERT_SHAP_VIEW = "_doc_abstract_only"

def scibert_predict_proba_shap(texts):
    out_probs = []

    model.eval()

    for text in texts:
        enc = tokenizer(
            str(text) if text is not None else "",
            truncation=True,
            max_length=MAX_LEN,
            padding="max_length",
            return_tensors="pt"
        )

        input_ids = enc["input_ids"].to(DEVICE)
        attention_mask = enc["attention_mask"].to(DEVICE)

        with torch.no_grad():
            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            ).logits

            probs = F.softmax(logits, dim=-1).detach().cpu().numpy()[0]

        out_probs.append(probs)

    return np.asarray(out_probs)


scibert_source = test_shap.copy()

if (scibert_source[COLS["label"]] == 1).any():
    scibert_source = scibert_source[scibert_source[COLS["label"]] == 1].copy()

scibert_sample = scibert_source.sample(min(5, len(scibert_source)), random_state=SEED)

texts_scibert = (
    scibert_sample[SCIBERT_SHAP_VIEW]
    .fillna("")
    .astype(str)
    .tolist()
)

texts_scibert = [t for t in texts_scibert if t.strip()]

scibert_shap_rows = []

if len(texts_scibert) > 0:
    masker_scibert = shap.maskers.Text(tokenizer)

    shap_explainer_scibert = shap.Explainer(
        scibert_predict_proba_shap,
        masker_scibert,
        output_names=["control", "retracted"]
    )

    shap_values_scibert = shap_explainer_scibert(texts_scibert)

    for doc_i, text in enumerate(texts_scibert):
        tokens = shap_values_scibert.data[doc_i]
        vals = shap_values_scibert.values[doc_i]

        if vals.ndim == 2:
            class_vals = vals[:, 1]
        else:
            class_vals = vals

        for tok, val in zip(tokens, class_vals):
            tok = str(tok).strip()

            if not tok:
                continue

            scibert_shap_rows.append({
                "model_family": "SciBERT",
                "doc_index": doc_i,
                "text_col": SCIBERT_SHAP_VIEW,
                "token": tok,
                "shap_value_class_1": float(val),
                "abs_shap": float(abs(val))
            })

scibert_shap_export = (
    pd.DataFrame(scibert_shap_rows).sort_values("abs_shap", ascending=False)
    if len(scibert_shap_rows)
    else pd.DataFrame()
)

shap_scibert_path = os.path.join(
    IG_EXPORT_DIR,
    "shap_local_scibert_tokens.csv"
)

scibert_shap_export.to_csv(shap_scibert_path, index=False)

print("Saved SciBERT SHAP:", shap_scibert_path)


# -----------------------------
# C. SHAP for Longformer
# -----------------------------

def longformer_predict_proba_shap_factory(lf_model, lf_tokenizer):
    def _predict(texts):
        out_probs = []

        lf_model.eval()

        for text in texts:
            enc = lf_tokenizer(
                str(text) if text is not None else "",
                truncation=True,
                max_length=LONG_MAX_LEN,
                padding="max_length",
                return_tensors="pt"
            )

            input_ids = enc["input_ids"].to(DEVICE)
            attention_mask = enc["attention_mask"].to(DEVICE)

            with torch.no_grad():
                logits = lf_model(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                ).logits

                probs = F.softmax(logits, dim=-1).detach().cpu().numpy()[0]

            out_probs.append(probs)

        return np.asarray(out_probs)

    return _predict


longformer_shap_rows = []

if "LONGFORMER_CKPTS" in globals() and len(LONGFORMER_CKPTS) > 0:

    for lf_view, lf_ckpt in LONGFORMER_CKPTS.items():
        print("Running Longformer SHAP for:", lf_view)

        lf_tokenizer = AutoTokenizer.from_pretrained(lf_ckpt)
        lf_model = AutoModelForSequenceClassification.from_pretrained(lf_ckpt).to(DEVICE)

        lf_source = test_shap.copy()

        if (lf_source[COLS["label"]] == 1).any():
            lf_source = lf_source[lf_source[COLS["label"]] == 1].copy()

        lf_sample = lf_source.sample(min(2, len(lf_source)), random_state=SEED)

        texts_lf = lf_sample[lf_view].fillna("").astype(str).tolist()
        texts_lf = [t for t in texts_lf if t.strip()]

        if len(texts_lf) == 0:
            del lf_model, lf_tokenizer
            _cleanup_cuda()
            continue

        pred_fn_lf = longformer_predict_proba_shap_factory(lf_model, lf_tokenizer)

        masker_lf = shap.maskers.Text(lf_tokenizer)

        shap_explainer_lf = shap.Explainer(
            pred_fn_lf,
            masker_lf,
            output_names=["control", "retracted"]
        )

        shap_values_lf = shap_explainer_lf(texts_lf)

        for doc_i, text in enumerate(texts_lf):
            tokens = shap_values_lf.data[doc_i]
            vals = shap_values_lf.values[doc_i]

            if vals.ndim == 2:
                class_vals = vals[:, 1]
            else:
                class_vals = vals

            for tok, val in zip(tokens, class_vals):
                tok = str(tok).strip()

                if not tok:
                    continue

                longformer_shap_rows.append({
                    "model_family": "Longformer",
                    "doc_index": doc_i,
                    "text_col": lf_view,
                    "token": tok,
                    "shap_value_class_1": float(val),
                    "abs_shap": float(abs(val))
                })

        del lf_model, lf_tokenizer, shap_explainer_lf, shap_values_lf
        _cleanup_cuda()

longformer_shap_export = (
    pd.DataFrame(longformer_shap_rows).sort_values("abs_shap", ascending=False)
    if len(longformer_shap_rows)
    else pd.DataFrame()
)

shap_longformer_path = os.path.join(
    IG_EXPORT_DIR,
    "shap_local_longformer_tokens.csv"
)

longformer_shap_export.to_csv(shap_longformer_path, index=False)

print("Saved Longformer SHAP:", shap_longformer_path)

display(pd.concat(
    [
        shap_global.head(10),
        scibert_shap_export.head(10),
        longformer_shap_export.head(10)
    ],
    ignore_index=True
))

In [ ]:
# CELL 49) Calibration across selected views with ECE and reliability data

def expected_calibration_error(y_true, p, n_bins=10):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)

    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0

    for lo, hi in zip(bins[:-1], bins[1:]):
        if hi < 1:
            m = (p >= lo) & (p < hi)
        else:
            m = (p >= lo) & (p <= hi)

        if m.any():
            ece += m.mean() * abs(y_true[m].mean() - p[m].mean())

    return float(ece)


def calibration_rows_for_col(text_col, agg="mean", batch_docs=2):
    val_ids, yv, pv = get_doc_probs_for_col(
        val_df,
        text_col,
        agg=agg,
        batch_docs=batch_docs
    )

    test_ids, yt, pt = get_doc_probs_for_col(
        test_df,
        text_col,
        agg=agg,
        batch_docs=batch_docs
    )

    platt = fit_platt_scaler(pv, yv)
    iso = fit_isotonic_scaler(pv, yv)

    variants = {
        "uncalibrated": pt,
        "platt": apply_platt_scaler(platt, pt),
        "isotonic": apply_isotonic_scaler(iso, pt),
    }

    rows = []

    for name, probs in variants.items():
        m = eval_probs(yt, probs)

        rows.append({
            "text_col": text_col,
            "agg": agg,
            "calibration": name,
            "AUPRC": m["AUPRC"],
            "ROC_AUC": m["ROC_AUC"],
            "Brier": m["Brier"],
            "ECE_10": expected_calibration_error(yt, probs, n_bins=10),
            "n_test_docs": int(len(yt)),
        })

    return pd.DataFrame(rows)


CALIBRATION_TEXT_COLS = [
    c for c in [
        "_doc_full",
        "_doc_abstract_only",
        "_doc_discussion_only",
        "_doc_methods_only",
        "_doc_core"
    ]
    if c in analysis_df.columns
]

cal_all = pd.concat(
    [
        calibration_rows_for_col(c, agg="mean", batch_docs=2)
        for c in CALIBRATION_TEXT_COLS
    ],
    ignore_index=True
)

cal_all_path = os.path.join(EXP_DIR, "calibration_all_views_with_ece.csv")
cal_all.to_csv(cal_all_path, index=False)

CAL_FINAL_DIR = os.path.join(RUN_DIR, "calibration_FINAL")
os.makedirs(CAL_FINAL_DIR, exist_ok=True)

cal_all.to_csv(
    os.path.join(CAL_FINAL_DIR, "calibration_all_views_with_ece.csv"),
    index=False
)

cal_all.to_csv(
    os.path.join(CAL_FINAL_DIR, "ece_metrics.csv"),
    index=False
)

with open(os.path.join(CAL_FINAL_DIR, "ece_metrics.json"), "w") as f:
    json.dump(cal_all.to_dict(orient="records"), f, indent=2)

# Reliability curve data for main full/mean model
_, yt_full, pt_full = get_doc_probs_for_col(
    test_df,
    "_doc_full",
    agg="mean",
    batch_docs=2
)

frac_pos, mean_pred = calibration_curve(
    yt_full,
    pt_full,
    n_bins=10,
    strategy="quantile"
)

reliability_df = pd.DataFrame({
    "mean_predicted_probability": mean_pred,
    "fraction_positive": frac_pos
})

reliability_df.to_csv(
    os.path.join(CAL_FINAL_DIR, "calibration_curves.csv"),
    index=False
)

print("Saved:", cal_all_path)
print("Calibration final artefacts saved to:", CAL_FINAL_DIR)
display(cal_all)

In [ ]:
# CELL 50) Extended temporal evaluation across model families and document views

TEMP_DIR = os.path.join(EXP_DIR, "temporal_extended")
os.makedirs(TEMP_DIR, exist_ok=True)

if "_split_time" not in analysis_df.columns:
    raise RuntimeError("Run Cell 37 first to create _split_time.")

known = analysis_df[
    analysis_df["_split_time"].isin(["train_time", "test_time"])
].copy()

if len(known) < 200:
    raise RuntimeError(f"Too few known-year rows for temporal evaluation: {len(known)}")

time_train_all = known[known["_split_time"] == "train_time"].copy()
time_test_df = known[known["_split_time"] == "test_time"].copy()

g_time = (
    time_train_all
    .groupby("_group_key", as_index=False)
    .agg({COLS["label"]: "first"})
)

sss = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

tr_i, va_i = next(sss.split(g_time["_group_key"], g_time[COLS["label"]]))

train_keys = set(g_time.iloc[tr_i]["_group_key"].astype(str))
val_keys = set(g_time.iloc[va_i]["_group_key"].astype(str))

time_train_df = time_train_all[
    time_train_all["_group_key"].astype(str).isin(train_keys)
].copy()

time_val_df = time_train_all[
    time_train_all["_group_key"].astype(str).isin(val_keys)
].copy()

print("Temporal train docs:", len(time_train_df))
print("Temporal val docs:", len(time_val_df))
print("Temporal future/test docs:", len(time_test_df))


def expected_calibration_error_local(y_true, p, n_bins=10):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)

    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0

    for lo, hi in zip(bins[:-1], bins[1:]):
        if hi < 1:
            m = (p >= lo) & (p < hi)
        else:
            m = (p >= lo) & (p <= hi)

        if m.any():
            ece += m.mean() * abs(y_true[m].mean() - p[m].mean())

    return float(ece)


def fit_platt_scaler_local(p_val, y_val):
    lr = SkLogisticRegression(max_iter=1000)
    lr.fit(np.asarray(p_val).reshape(-1, 1), np.asarray(y_val).astype(int))
    return lr


def apply_platt_scaler_local(model_cal, p):
    return model_cal.predict_proba(np.asarray(p).reshape(-1, 1))[:, 1]


def fit_isotonic_scaler_local(p_val, y_val):
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(np.asarray(p_val), np.asarray(y_val).astype(int))
    return iso


def apply_isotonic_scaler_local(iso, p):
    return iso.predict(np.asarray(p))


TEMPORAL_TEXT_COLS = [
    c for c in [
        "_doc_full",
        "_doc_core",
        "_doc_abstract_only",
        "_doc_methods_only",
        "_doc_results_only",
        "_doc_discussion_only",
    ]
    if c in analysis_df.columns
]

temporal_rows = []


# -----------------------------------------
# A. SciBERT static checkpoint on temporal split
# -----------------------------------------

for text_col in TEMPORAL_TEXT_COLS:
    _, yv, pv = get_doc_probs_for_col(
        time_val_df,
        text_col,
        agg="mean",
        batch_docs=2
    )

    _, yt, pt = get_doc_probs_for_col(
        time_test_df,
        text_col,
        agg="mean",
        batch_docs=2
    )

    platt = fit_platt_scaler_local(pv, yv)
    iso = fit_isotonic_scaler_local(pv, yv)

    variants = {
        "uncalibrated": pt,
        "platt": apply_platt_scaler_local(platt, pt),
        "isotonic": apply_isotonic_scaler_local(iso, pt),
    }

    for cal_name, probs in variants.items():
        m = eval_probs(yt, probs)

        temporal_rows.append({
            "model_family": "SciBERT_static",
            "text_col": text_col,
            "feature_type": "chunked_transformer_mean",
            "calibration": cal_name,
            "AUPRC": m["AUPRC"],
            "ROC_AUC": m["ROC_AUC"],
            "Brier": m["Brier"],
            "ECE_10": expected_calibration_error_local(yt, probs, n_bins=10),
            "n_val_docs": int(len(yv)),
            "n_future_docs": int(len(yt)),
        })


# -----------------------------------------
# B. TF-IDF word and char n-grams on temporal split
# -----------------------------------------

def temporal_tfidf_eval(text_col, analyzer, ngram, feature_label, max_features=None):
    X_train = time_train_df[text_col].fillna("")
    y_train = time_train_df[COLS["label"]].astype(int)

    X_val = time_val_df[text_col].fillna("")
    y_val = time_val_df[COLS["label"]].astype(int)

    X_test = time_test_df[text_col].fillna("")
    y_test = time_test_df[COLS["label"]].astype(int)

    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(
            analyzer=analyzer,
            ngram_range=ngram,
            min_df=3,
            max_df=0.95,
            max_features=max_features,
            strip_accents="unicode",
            lowercase=True
        )),
        ("clf", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            n_jobs=-1
        ))
    ])

    pipe.fit(X_train, y_train)

    pv = pipe.predict_proba(X_val)[:, 1]
    pt = pipe.predict_proba(X_test)[:, 1]

    platt = fit_platt_scaler_local(pv, y_val)
    iso = fit_isotonic_scaler_local(pv, y_val)

    variants = {
        "uncalibrated": pt,
        "platt": apply_platt_scaler_local(platt, pt),
        "isotonic": apply_isotonic_scaler_local(iso, pt),
    }

    rows = []

    for cal_name, probs in variants.items():
        m = eval_probs(y_test, probs)

        rows.append({
            "model_family": "TFIDF+LogReg",
            "text_col": text_col,
            "feature_type": feature_label,
            "calibration": cal_name,
            "AUPRC": m["AUPRC"],
            "ROC_AUC": m["ROC_AUC"],
            "Brier": m["Brier"],
            "ECE_10": expected_calibration_error_local(y_test, probs, n_bins=10),
            "n_val_docs": int(len(y_val)),
            "n_future_docs": int(len(y_test)),
        })

    return rows


for text_col in TEMPORAL_TEXT_COLS:
    temporal_rows.extend(
        temporal_tfidf_eval(
            text_col=text_col,
            analyzer="word",
            ngram=(1, 2),
            feature_label="word_1_2"
        )
    )

    temporal_rows.extend(
        temporal_tfidf_eval(
            text_col=text_col,
            analyzer="char_wb",
            ngram=(3, 5),
            feature_label="char_wb_3_5",
            max_features=300000
        )
    )


# -----------------------------------------
# C. Longformer static checkpoint on temporal split
# -----------------------------------------

if "LONGFORMER_CKPTS" in globals() and len(LONGFORMER_CKPTS) > 0:

    for lf_view, lf_ckpt in LONGFORMER_CKPTS.items():

        if lf_view not in time_val_df.columns:
            continue

        print("Temporal Longformer evaluation for:", lf_view)

        lf_tokenizer = AutoTokenizer.from_pretrained(lf_ckpt)
        lf_model = AutoModelForSequenceClassification.from_pretrained(lf_ckpt).to(DEVICE)

        val_long_ds = LongTextDataset(
            time_val_df,
            lf_view,
            COLS["label"],
            COLS["paper_id"],
            lf_tokenizer,
            max_len=LONG_MAX_LEN
        )

        test_long_ds = LongTextDataset(
            time_test_df,
            lf_view,
            COLS["label"],
            COLS["paper_id"],
            lf_tokenizer,
            max_len=LONG_MAX_LEN
        )

        val_long_dl = DataLoader(
            val_long_ds,
            batch_size=1,
            shuffle=False,
            collate_fn=collate_longformer,
            num_workers=0
        )

        test_long_dl = DataLoader(
            test_long_ds,
            batch_size=1,
            shuffle=False,
            collate_fn=collate_longformer,
            num_workers=0
        )

        _, yv_l, pv_l = predict_longformer_probs(lf_model, val_long_dl)
        _, yt_l, pt_l = predict_longformer_probs(lf_model, test_long_dl)

        platt_l = fit_platt_scaler_local(pv_l, yv_l)
        iso_l = fit_isotonic_scaler_local(pv_l, yv_l)

        variants_l = {
            "uncalibrated": pt_l,
            "platt": apply_platt_scaler_local(platt_l, pt_l),
            "isotonic": apply_isotonic_scaler_local(iso_l, pt_l),
        }

        for cal_name, probs in variants_l.items():
            m = eval_probs(yt_l, probs)

            temporal_rows.append({
                "model_family": "Longformer_static",
                "text_col": lf_view,
                "feature_type": f"long_context_{LONG_MAX_LEN}",
                "calibration": cal_name,
                "AUPRC": m["AUPRC"],
                "ROC_AUC": m["ROC_AUC"],
                "Brier": m["Brier"],
                "ECE_10": expected_calibration_error_local(yt_l, probs, n_bins=10),
                "n_val_docs": int(len(yv_l)),
                "n_future_docs": int(len(yt_l)),
            })

        del lf_model, lf_tokenizer, val_long_ds, test_long_ds, val_long_dl, test_long_dl
        _cleanup_cuda()
else:
    print("No Longformer checkpoints found. Skipping Longformer temporal evaluation.")


temporal_extended = pd.DataFrame(temporal_rows)

temporal_extended_path = os.path.join(
    TEMP_DIR,
    "temporal_evaluation_across_models_views.csv"
)

temporal_extended_json_path = os.path.join(
    TEMP_DIR,
    "temporal_evaluation_extended.json"
)

temporal_extended.to_csv(temporal_extended_path, index=False)

save_json(
    {
        "temporal_train_docs": int(len(time_train_df)),
        "temporal_val_docs": int(len(time_val_df)),
        "temporal_future_docs": int(len(time_test_df)),
        "results_csv": temporal_extended_path,
        "results": temporal_extended.to_dict(orient="records"),
    },
    temporal_extended_json_path
)

shutil.copy(
    temporal_extended_json_path,
    os.path.join(RUN_DIR, "temporal_calibration.json")
)

print("Saved extended temporal evaluation:", temporal_extended_path)
print("Saved extended temporal JSON:", temporal_extended_json_path)

display(
    temporal_extended.sort_values(
        ["model_family", "text_col", "feature_type", "calibration"]
    )
)

In [ ]:
# CELL 51) Freeze temporal evaluation

TEMPORAL_FINAL = os.path.join(RUN_DIR, "temporal_eval_FINAL.json")

candidates = [
    os.path.join(EXP_DIR, "temporal_extended", "temporal_evaluation_extended.json"),
    os.path.join(EXP_DIR, "temporal_eval_scibert_RETRAIN.json"),
    os.path.join(EXP_DIR, "temporal_eval_scibert.json"),
    os.path.join(EXP_DIR, "temporal_eval.json"),
    os.path.join(RUN_DIR, "temporal_eval_scibert.json"),
    os.path.join(RUN_DIR, "temporal_eval.json"),
]

hits = glob.glob(os.path.join(EXP_DIR, "**", "temporal_eval*.json"), recursive=True)
hits += glob.glob(os.path.join(EXP_DIR, "**", "temporal_evaluation_extended.json"), recursive=True)

for h in sorted(hits, key=os.path.getmtime, reverse=True):
    if h not in candidates:
        candidates.append(h)

src = next((p for p in candidates if os.path.exists(p)), None)

if src is None:
    raise FileNotFoundError(
        "Temporal evaluation JSON not found. Searched:\n- " + "\n- ".join(candidates)
    )

shutil.copy(src, TEMPORAL_FINAL)

print("Temporal evaluation frozen:")
print("   src:", src)
print("   dst:", TEMPORAL_FINAL)

with open(TEMPORAL_FINAL, "r") as f:
    temporal_data = json.load(f)

print("Keys in temporal_eval_FINAL.json:", list(temporal_data.keys()))

In [ ]:
# CELL 52) Explanation stability over time using SciBERT IG faithfulness

EXPLANATION_STABILITY_TEXT_COLS = [
    c for c in [
        "_doc_abstract_only",
        "_doc_methods_only",
        "_doc_results_only",
        "_doc_discussion_only",
    ]
    if c in analysis_df.columns
]

if "_split_time" not in analysis_df.columns:
    raise RuntimeError("Run Cell 37 first.")

historical_df = analysis_df[analysis_df["_split_time"] == "train_time"].copy()
future_df = analysis_df[analysis_df["_split_time"] == "test_time"].copy()

if len(historical_df) == 0 or len(future_df) == 0:
    raise RuntimeError("Temporal historical/future split is empty. Check Cell 37.")

TEMP_DIR = os.path.join(EXP_DIR, "temporal_extended")
os.makedirs(TEMP_DIR, exist_ok=True)

stability_parts = []

for text_col in EXPLANATION_STABILITY_TEXT_COLS:
    print("Running explanation stability audit for:", text_col)

    hist_audit = faithfulness_audit(
        historical_df,
        text_col=text_col,
        n=20,
        k_fracs=(0.05, 0.10),
        n_steps=8,
        seed=SEED
    )

    hist_audit["period"] = "historical_train_time"
    hist_audit["text_col"] = text_col

    fut_audit = faithfulness_audit(
        future_df,
        text_col=text_col,
        n=20,
        k_fracs=(0.05, 0.10),
        n_steps=8,
        seed=SEED
    )

    fut_audit["period"] = "future_test_time"
    fut_audit["text_col"] = text_col

    stability_parts.extend([hist_audit, fut_audit])

if len(stability_parts) == 0:
    raise RuntimeError("No explanation stability audits were generated.")

stability = pd.concat(stability_parts, ignore_index=True)

stability_path = os.path.join(
    TEMP_DIR,
    "explanation_stability_historical_vs_future.csv"
)

stability.to_csv(stability_path, index=False)

delta_cols = [c for c in stability.columns if c.startswith("delta_")]

if len(delta_cols) == 0:
    raise RuntimeError("No delta_* columns found in explanation stability output.")

stability_summary = (
    stability
    .groupby(["period", "text_col"])[delta_cols]
    .mean()
    .reset_index()
)

stability_summary_path = os.path.join(
    TEMP_DIR,
    "explanation_stability_summary.csv"
)

stability_summary.to_csv(stability_summary_path, index=False)

shutil.copy(
    stability_path,
    os.path.join(RUN_DIR, "explanation_stability.csv")
)

shutil.copy(
    stability_summary_path,
    os.path.join(RUN_DIR, "explanation_stability_summary.csv")
)

print("Saved:", stability_path)
print("Saved:", stability_summary_path)
print("Copied to:", os.path.join(RUN_DIR, "explanation_stability.csv"))
print("Copied to:", os.path.join(RUN_DIR, "explanation_stability_summary.csv"))

display(stability_summary)

PART K — Methodology coverage and final artefact checklist

In [ ]:
# CELL 53) Check implementation coverage against Methodology / Proposal

IMPLEMENTATION_CHECK = pd.DataFrame([
    {
        "methodology_item": "TF-IDF + Logistic Regression baseline with word n-grams",
        "status": "implemented",
        "where": "Cells 8 and 38"
    },
    {
        "methodology_item": "TF-IDF + Logistic Regression baseline with character n-grams",
        "status": "implemented",
        "where": "Cell 38"
    },
    {
        "methodology_item": "SciBERT section-aware modelling",
        "status": "implemented",
        "where": "Cells 9–18 and 25–29"
    },
    {
        "methodology_item": "Long-context transformer model family across document views",
        "status": "implemented",
        "where": "Cells 11 and 30"
    },
    {
        "methodology_item": "Section masking / ablation",
        "status": "implemented",
        "where": "Cells 15, 17–18, 27–29, and 42"
    },
    {
        "methodology_item": "Integrated Gradients token attribution for SciBERT",
        "status": "implemented",
        "where": "Cells 19–24, 43, 45–46, and 52"
    },
    {
        "methodology_item": "LIME local explanations for TF-IDF, SciBERT, and Longformer",
        "status": "implemented",
        "where": "Cell 47"
    },
    {
        "methodology_item": "SHAP explanations for TF-IDF, SciBERT, and Longformer",
        "status": "implemented",
        "where": "Cell 48"
    },
    {
        "methodology_item": "Perturbation faithfulness / Delta P",
        "status": "implemented",
        "where": "Cells 21–24, 31–32, 46, and 52"
    },
    {
        "methodology_item": "Calibration: Platt scaling, isotonic regression, ECE, and reliability data",
        "status": "implemented",
        "where": "Cells 33–36, 44, and 49"
    },
    {
        "methodology_item": "Temporal generalisation across model families and document views",
        "status": "implemented",
        "where": "Cells 37, 50, and 51"
    },
    {
        "methodology_item": "Explanation stability over time",
        "status": "implemented",
        "where": "Cell 52"
    },
])

In [ ]:
# CELL 54) FINAL ARTEFACT CHECKLIST

artefacts = {
    "Final dataset": FINAL_DB_PATH,
    "Dataset metadata": os.path.join(RUN_DIR, "dataset_metadata.json"),
    "Model registry": os.path.join(RUN_DIR, "model_registry.json"),

    "Final results table": FINAL_RESULTS_PATH,
    "Section ablation table": MASK_TABLE_PATH,

    "XAI example (IG)": XAI_SINGLE_FINAL,
    "XAI faithfulness summary": XAI_SUMMARY_FINAL,

    "IG token-level outputs": os.path.join(
        EXP_DIR,
        "xai_exports",
        "integrated_gradients_top_tokens.csv"
    ),

    "LIME TF-IDF": os.path.join(
        EXP_DIR,
        "xai_exports",
        "lime_local_explanations_tfidf.csv"
    ),

    "LIME SciBERT": os.path.join(
        EXP_DIR,
        "xai_exports",
        "lime_local_explanations_scibert.csv"
    ),

    "LIME Longformer": os.path.join(
        EXP_DIR,
        "xai_exports",
        "lime_local_explanations_longformer.csv"
    ),

    "SHAP global TF-IDF terms": os.path.join(
        EXP_DIR,
        "xai_exports",
        "shap_global_tfidf_top_terms.csv"
    ),

    "SHAP SciBERT": os.path.join(
        EXP_DIR,
        "xai_exports",
        "shap_local_scibert_tokens.csv"
    ),

    "SHAP Longformer": os.path.join(
        EXP_DIR,
        "xai_exports",
        "shap_local_longformer_tokens.csv"
    ),

    "Calibration metrics": os.path.join(
        CAL_FINAL_DIR,
        "calibration_metrics_FINAL.json"
    ),

    "Calibration all views + ECE": os.path.join(
        CAL_FINAL_DIR,
        "calibration_all_views_with_ece.csv"
    ),

    "Calibration curves": os.path.join(
        CAL_FINAL_DIR,
        "calibration_curves.csv"
    ),

    "ECE metrics JSON": os.path.join(
        CAL_FINAL_DIR,
        "ece_metrics.json"
    ),

    "Temporal robustness": TEMPORAL_FINAL,

    "Temporal calibration": os.path.join(
        RUN_DIR,
        "temporal_calibration.json"
    ),

    "Longformer full checkpoint": os.path.join(
        RUN_DIR,
        "ckpt_longformer_full"
    ),

    "Longformer temporal / view results": os.path.join(
        TEMP_DIR,
        "temporal_evaluation_across_models_views.csv"
    ),

    "TF-IDF word/char summary": os.path.join(
        EXP_DIR,
        "classical_tfidf_word_char_summary.csv"
    ),

    "Explanation stability": os.path.join(
        RUN_DIR,
        "explanation_stability.csv"
    ),

    "Explanation stability summary": os.path.join(
        RUN_DIR,
        "explanation_stability_summary.csv"
    ),
}

print("=== FINAL ARTEFACTS ===")

missing = []

for k, v in artefacts.items():
    exists = os.path.exists(v)
    print(f"{'✔' if exists else '✘'} {k}: {exists}  → {v}")

    if not exists:
        missing.append(k)

if missing:
    print("\nMissing artefacts:", missing)
else:
    print("\nAll artefacts present — pipeline complete.")